## Total Forecasting 

### 초반 설정 (패키지 설치)

In [2]:
%pip install pandas numpy holidays scikit-learn xgboost pymysql sshtunnel python-dotenv openpyxl requests tqdm urllib3 ipykernel 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
!pip uninstall -y paramiko
!pip install paramiko==3.5.1 sshtunnel==0.4.0

Found existing installation: paramiko 3.5.1
Uninstalling paramiko-3.5.1:
  Successfully uninstalled paramiko-3.5.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached paramiko-3.5.1-py3-none-any.whl.metadata (4.6 kB)
Using cached paramiko-3.5.1-py3-none-any.whl (227 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import sys
print(sys.executable)

C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


In [4]:
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import requests
import numpy as np
from tqdm import tqdm

print("import 성공")

import 성공


In [3]:
pip install "paramiko<4"

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: paramiko
    Found existing installation: paramiko 4.0.0
    Uninstalling paramiko-4.0.0:
      Successfully uninstalled paramiko-4.0.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Data load & Update

In [ ]:
# ==========================================================
# Cell 1. 데이터 로드 + 기온 병합 + 5-Track 타겟 생성
# ==========================================================

import os
import numpy as np
import pandas as pd
import pymysql
import requests
import urllib3

from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from tqdm import tqdm
from datetime import datetime, timedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv()


# ==========================================================
# 1. DB 데이터 로드
# ==========================================================
class AutoContainerGeneration:
    def __init__(self, version=7, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

    def fetch_data(
        self,
        query,
        ssh_host,
        ssh_user,
        ssh_private_key,
        mysql_host,
        mysql_port,
        mysql_user,
        mysql_password,
        mysql_database,
    ):
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port),
            ) as tunnel:
                print("SSH 터널 연결 성공")

                with pymysql.connect(
                    host="127.0.0.1",
                    user=mysql_user,
                    passwd=mysql_password,
                    db=mysql_database,
                    charset="utf8",
                    port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor,
                ) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")
                        return pd.DataFrame(results)

        except Exception as e:
            print(f"Error fetching data: {e}")
            return None

    def fetch_cluster_data(self):
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv("SSH_PRIVATE_KEY")

        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        print(f"Environment: {mysql_host}")

        # v.7 변경점:
        # - 지그재그는 당일/익일 유지
        # - 카카오모빌리티, 부스터스는 독립 track으로 추출
        # - 기타는 더 이상 기타_당일/기타_익일로 분리하지 않음
        # - 순수_기타물량은 Python에서 residual로 생성
        cluster_query = """
            SELECT
                hub_cluster_plan.plan_date AS 날짜,

                COUNT(DISTINCT shipping_shippingitem.id) AS "총 물량",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'sameday'
                        OR (
                            JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL
                            AND HOUR(order_order.`timestamp`) < 13
                        )
                    )
                    THEN shipping_shippingitem.id
                END) AS "지그재그_당일",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'nextday'
                        OR (
                            JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL
                            AND HOUR(order_order.`timestamp`) >= 13
                        )
                    )
                    THEN shipping_shippingitem.id
                END) AS "지그재그_익일",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '카카오모빌리티'
                    THEN shipping_shippingitem.id
                END) AS "카카오모빌리티",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '부스터스'
                    THEN shipping_shippingitem.id
                END) AS "부스터스",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '지그재그 당일배송'
                    THEN shipping_shippingitem.id
                END) AS "지그재그 당일배송",

                COUNT(DISTINCT shop_shop.name) AS "셀러수"

            FROM shipping_shippingitem
            JOIN order_order
                ON shipping_shippingitem.order_id = order_order.id
            JOIN hub_cluster_plan_item
                ON shipping_shippingitem.id = hub_cluster_plan_item.shipping_item_id
            JOIN hub_cluster_plan
                ON hub_cluster_plan_item.cluster_plan_id = hub_cluster_plan.id
            JOIN location_address la
                ON shipping_shippingitem.address_id = la.id
            JOIN location_sector ls
                ON shipping_shippingitem.designated_sector_id = ls.id
            JOIN shop_shop
                ON shop_shop.id = order_order.shop_id

            WHERE hub_cluster_plan.plan_date <= CURRENT_DATE()
              AND ls.allowed = 1

            GROUP BY hub_cluster_plan.plan_date
            ORDER BY hub_cluster_plan.plan_date;
        """

        return self.fetch_data(
            cluster_query,
            ssh_host,
            ssh_user,
            ssh_private_key,
            mysql_host,
            mysql_port,
            mysql_user,
            mysql_password,
            mysql_database,
        )


# ==========================================================
# 2. 기존 마스터 데이터 + DB 최신 데이터 병합
# ==========================================================
generator = AutoContainerGeneration(version=7, debug=True)
df_forecast = generator.fetch_cluster_data()

if df_forecast is None or df_forecast.empty:
    print("🚨 DB에서 데이터를 가져오지 못했습니다. 기존 ./data/total_latest.xlsx만 로드합니다.")
    data = pd.read_excel("./data/total_latest.xlsx")
else:
    print("✅ 최신 DB 데이터 추출 완료")

    db_df = df_forecast.copy()
    db_df.columns = [c.replace("총_물량", "총 물량") for c in db_df.columns]
    db_df["날짜"] = pd.to_datetime(db_df["날짜"], errors="coerce").dt.normalize()

    try:
        base = pd.read_excel("./data/total_latest.xlsx")
        base["날짜"] = pd.to_datetime(base["날짜"], errors="coerce").dt.normalize()
    except FileNotFoundError:
        base = pd.DataFrame()

    data = pd.concat([base, db_df], axis=0).reset_index(drop=True)
    data = data.sort_values("날짜")
    data = data.drop_duplicates(subset=["날짜"], keep="last").reset_index(drop=True)


# ==========================================================
# 3. 5-Track 타겟 생성
# ==========================================================
required_cols = [
    "날짜",
    "총 물량",
    "지그재그_당일",
    "지그재그_익일",
    "카카오모빌리티",
    "부스터스",
    "지그재그 당일배송",
    "셀러수",
]

for col in required_cols:
    if col not in data.columns:
        data[col] = 0

data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()

numeric_cols = [
    "총 물량",
    "지그재그_당일",
    "지그재그_익일",
    "카카오모빌리티",
    "부스터스",
    "지그재그 당일배송",
    "셀러수",
]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0)

# 핵심 변경:
# 순수_기타물량 = 총 물량 - 지그재그_당일 - 지그재그_익일 - 카카오 - 부스터스
data["순수_기타물량"] = (
    data["총 물량"]
    - data["지그재그_당일"]
    - data["지그재그_익일"]
    - data["카카오모빌리티"]
    - data["부스터스"]
).clip(lower=0)

final_cols = [
    "날짜",
    "총 물량",
    "지그재그_당일",
    "지그재그_익일",
    "카카오모빌리티",
    "부스터스",
    "순수_기타물량",
    "지그재그 당일배송",
    "셀러수",
]

data = data[final_cols].sort_values("날짜").reset_index(drop=True)

data.to_excel("./data/total_latest.xlsx", index=False)
print("✨ 5-Track 마스터 데이터 저장 완료")
display(data.tail(10))


# ==========================================================
# 4. 기온 데이터 로드/갱신/병합
# ==========================================================

# 보안상 API 키는 코드에 직접 쓰지 말고 .env에 KMA_API_KEY로 넣는 것을 권장
serviceKey = os.getenv("KMA_API_KEY")

if serviceKey is None:
    print("⚠️ KMA_API_KEY가 .env에 없습니다. 기존 recent_temp.xlsx만 사용합니다.")


def fetch_short_term_forecast(base_date: str, base_time: str, region_point: dict):
    records_all = []

    if serviceKey is None:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    for region, (nx, ny) in tqdm(region_point.items(), desc="단기예보 수집"):
        url = (
            "https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getVilageFcst"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&base_date={base_date}&base_time={base_time}"
            f"&nx={nx}&ny={ny}&authKey={serviceKey}"
        )

        try:
            response = requests.get(url, verify=False, timeout=20)
            response.raise_for_status()
            json_response = response.json()
            items = json_response["response"]["body"]["items"]["item"]
        except Exception as e:
            print(f"⚠️ {region} 단기 데이터 요청 실패: {e}")
            continue

        informations = {}
        for it in items:
            cate = it["category"]
            fcstDate = it["fcstDate"]
            fcstTime = it["fcstTime"]
            fcstValue = it["fcstValue"]
            key = f"{fcstDate}_{fcstTime}"

            if key not in informations:
                informations[key] = {}
            informations[key][cate] = fcstValue

        region_records = []
        for key, vals in informations.items():
            if "TMP" in vals:
                date = key.split("_")[0]
                region_records.append({"date": date, "TMP": float(vals["TMP"])})

        if not region_records:
            continue

        df_region = (
            pd.DataFrame(region_records)
            .groupby("date", as_index=False)["TMP"]
            .mean()
            .rename(columns={"TMP": "avg_temp"})
        )
        df_region["지역"] = region
        records_all.append(df_region)

    if not records_all:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    df_all = pd.concat(records_all, ignore_index=True)
    df_all.rename(columns={"date": "날짜"}, inplace=True)
    df_all["날짜"] = pd.to_datetime(df_all["날짜"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    return df_all


def fetch_mid_term_forecast(base_date: str, region_code: dict):
    records = []

    if serviceKey is None:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    for region, regId in tqdm(region_code.items(), desc="중기예보 수집"):
        url = (
            "https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService/getMidTa"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&regId={regId}&tmFc={base_date}0600&authKey={serviceKey}"
        )

        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            item = response.json()["response"]["body"]["items"]["item"][0]
        except Exception as e:
            print(f"⚠️ {region}({regId}) 중기 데이터 요청 실패: {e}")
            continue

        for day in range(4, 11):
            ta_min = item.get(f"taMin{day}")
            ta_max = item.get(f"taMax{day}")

            if ta_min is not None and ta_max is not None:
                records.append(
                    {
                        "날짜": (datetime.now() + timedelta(days=day)).strftime("%Y-%m-%d"),
                        "지역": region,
                        "avg_temp": (ta_min + ta_max) / 2,
                    }
                )

    return pd.DataFrame(records, columns=["날짜", "지역", "avg_temp"])


region_point = {
    "서울특별시": ("61", "126"),
    "인천광역시": ("54", "125"),
    "경기도": ("60", "121"),
    "대전광역시": ("68", "100"),
    "충청남도": ("65", "110"),
}

region_code = {
    "서울특별시": "11B10101",
    "인천광역시": "11B20201",
    "경기도": "11B20601",
    "대전광역시": "11C20401",
    "충청남도": "11C20101",
}

short_df = fetch_short_term_forecast(
    base_date=datetime.now().strftime("%Y%m%d"),
    base_time="0200",
    region_point=region_point,
)

mid_df = fetch_mid_term_forecast(
    base_date=datetime.now().strftime("%Y%m%d"),
    region_code=region_code,
)

combined_df = pd.concat([short_df, mid_df], ignore_index=True)

if not combined_df.empty:
    combined_df["날짜"] = pd.to_datetime(combined_df["날짜"]).dt.strftime("%Y-%m-%d")
    final_df = combined_df.drop_duplicates(subset=["날짜", "지역"], keep="first").reset_index(drop=True)
    final_df = final_df.sort_values(["지역", "날짜"]).reset_index(drop=True)
    final_df["avg_temp"] = np.round(final_df["avg_temp"], 1)
    final_df = final_df[["날짜", "지역", "avg_temp"]]
else:
    final_df = pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

try:
    base_temp = pd.read_excel("./data/recent_temp.xlsx")
    base_temp["날짜"] = pd.to_datetime(base_temp["날짜"], errors="coerce")
except FileNotFoundError:
    base_temp = pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

temp_data = pd.concat([base_temp, final_df], axis=0).reset_index(drop=True)
temp_data["날짜"] = pd.to_datetime(temp_data["날짜"], errors="coerce")
temp_data = temp_data.drop_duplicates(subset=["날짜", "지역"], keep="last").reset_index(drop=True)
temp_data = temp_data.sort_values(["지역", "날짜"]).reset_index(drop=True)
temp_data.to_excel("./data/recent_temp.xlsx", index=False)

temp_merge_data = temp_data.groupby("날짜", as_index=False)["avg_temp"].mean()
temp_merge_data["avg_temp"] = np.round(temp_merge_data["avg_temp"], 1)
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"], errors="coerce").dt.normalize()

data = pd.merge(data, temp_merge_data, how="left", on="날짜")
print("✅ 기온 데이터 병합 완료")
display(data.tail(10))

AutoContainerGeneration version: 7
Environment: prod-daas-dw-cluster.cluster-cgrjlsurd8vz.ap-northeast-2.rds.amazonaws.com
SSH 터널 연결 성공
쿼리 실행 완료
✅ 최신 DB 데이터 추출 완료
✨ 5-Track 마스터 데이터 저장 완료


,날짜,총 물량,지그재그_당일,지그재그_이월,카카오모빌리티,부스터스,순수_기타물량,지그재그 당일배송,셀러수
857,2026-05-09,11157,5320,5422,263,0,152,10742,7
858,2026-05-10,10583,5293,5009,162,0,119,10302,7
859,2026-05-11,19305,6459,6647,1431,2149,2619,13106,23
860,2026-05-12,17914,6703,7591,902,567,2151,14294,24
861,2026-05-13,21643,8287,9108,1262,834,2152,17395,25
862,2026-05-14,22482,7660,10616,1022,601,2583,18276,25
863,2026-05-15,22749,7855,11185,1037,325,2347,19040,26
864,2026-05-16,15618,7543,6955,348,0,772,14498,13
865,2026-05-17,13093,6881,5927,153,0,132,12808,10
866,2026-05-18,23810,8213,7964,1773,1114,4746,16177,23


⚠️ KMA_API_KEY가 .env에 없습니다. 기존 recent_temp.xlsx만 사용합니다.
✅ 기온 데이터 병합 완료


,날짜,총 물량,지그재그_당일,지그재그_이월,카카오모빌리티,부스터스,순수_기타물량,지그재그 당일배송,셀러수,avg_temp
857,2026-05-09,11157,5320,5422,263,0,152,10742,7,16.0
858,2026-05-10,10583,5293,5009,162,0,119,10302,7,16.9
859,2026-05-11,19305,6459,6647,1431,2149,2619,13106,23,17.9
860,2026-05-12,17914,6703,7591,902,567,2151,14294,24,18.1
861,2026-05-13,21643,8287,9108,1262,834,2152,17395,25,18.6
862,2026-05-14,22482,7660,10616,1022,601,2583,18276,25,20.8
863,2026-05-15,22749,7855,11185,1037,325,2347,19040,26,21.2
864,2026-05-16,15618,7543,6955,348,0,772,14498,13,18.8
865,2026-05-17,13093,6881,5927,153,0,132,12808,10,21.6
866,2026-05-18,23810,8213,7964,1773,1114,4746,16177,23,22.2


### Feature & Modeling

In [ ]:
# ==========================================================
# Cell 2. Feature + Model 함수 전체
# ==========================================================

import numpy as np
import pandas as pd
import holidays

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error


# ==========================================================
# 0. 5-Track 타겟 정의
# ==========================================================
ZIG_SAME = "지그재그_당일"
ZIG_NEXT = "지그재그_익일"
KAKAO = "카카오모빌리티"
BOOSTERS = "부스터스"
OTHER_PURE = "순수_기타물량"
TOTAL = "총 물량"

BASE_TARGETS = [
    ZIG_SAME,
    ZIG_NEXT,
    KAKAO,
    BOOSTERS,
    OTHER_PURE,
]

ALL_TARGETS = BASE_TARGETS + [TOTAL]


# ==========================================================
# 1. 공통 Feature 함수
# ==========================================================
def add_temp_shift_features(df, date_col="날짜", temp_col="avg_temp"):
    df = df.copy().sort_values(date_col)

    # ✅ avg_temp를 반드시 숫자로 변환
    df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")

    # 결측 보정
    df[temp_col] = (
        df[temp_col]
        .ffill()
        .bfill()
        .fillna(15.0)
    )

    temp_ma7_min = df[temp_col].rolling(7, min_periods=1).min()
    diff = temp_ma7_min - df[temp_col]

    df["temp_shift"] = diff.abs().astype(float)

    # ✅ np.sign 대신 안전한 방식
    df["temp_shift_sign"] = 0
    df.loc[diff > 0, "temp_shift_sign"] = 1
    df.loc[diff < 0, "temp_shift_sign"] = -1

    df[["temp_shift", "temp_shift_sign"]] = (
        df[["temp_shift", "temp_shift_sign"]]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return df


def add_target_lag_rolling_features(
    df,
    target_col,
    lags=(1, 7, 14, 21, 28),
    roll_windows=(7, 14, 21, 28),
):
    """
    Target leakage 방지:
    - lag는 shift(lag)
    - rolling은 반드시 shift(1) 이후 계산
    """
    df = df.copy().sort_values("날짜").reset_index(drop=True)

    if target_col not in df.columns:
        for lag in lags:
            df[f"{target_col}_lag_{lag}"] = np.nan
        for w in roll_windows:
            df[f"{target_col}_roll_mean_{w}"] = np.nan
            df[f"{target_col}_roll_std_{w}"] = np.nan
        return df

    s = pd.to_numeric(df[target_col], errors="coerce")

    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = s.shift(lag)

    shifted = s.shift(1)
    for w in roll_windows:
        df[f"{target_col}_roll_mean_{w}"] = shifted.rolling(
            w,
            min_periods=max(2, w // 4),
        ).mean()
        df[f"{target_col}_roll_std_{w}"] = shifted.rolling(
            w,
            min_periods=max(2, w // 4),
        ).std()

    return df


def build_holiday_set(years, manual_holidays=None):
    """
    holidays.KR 자동 공휴일 + 수동 공휴일을 하나의 set(date)로 통합.
    근로자의 날(5/1), 임시공휴일 등은 manual_holidays로 추가.
    """
    kr_holiday_set = set(holidays.KR(years=years).keys())

    if manual_holidays:
        manual_set = set(pd.to_datetime(manual_holidays).date)
        kr_holiday_set.update(manual_set)

    return kr_holiday_set


def add_pre_post_holiday_features(df, holiday_set):
    """
    공휴일 전후 효과를 생성.
    - pre_holiday_peak: 다음 1~2영업일 내 공휴일이 있으면 피크 가능성
    - post_holiday_peak: 직전 공휴일 이후 첫 영업일
    - days_until_holiday: 다음 공휴일까지 남은 일수
    - days_after_holiday: 직전 공휴일로부터 지난 일수
    """
    df = df.copy()
    df_dates = df["날짜"].dt.date.tolist()
    holiday_dates = sorted(list(holiday_set))

    holiday_ts = pd.to_datetime(holiday_dates)

    df["pre_holiday_peak"] = 0
    df["post_holiday_peak"] = 0
    df["days_until_holiday"] = 99
    df["days_after_holiday"] = 99

    if len(holiday_ts) == 0:
        return df

    date_series = df["날짜"]

    for idx, cur_date in enumerate(date_series):
        cur_norm = pd.Timestamp(cur_date).normalize()

        future_holidays = holiday_ts[holiday_ts >= cur_norm]
        past_holidays = holiday_ts[holiday_ts <= cur_norm]

        if len(future_holidays) > 0:
            days_until = int((future_holidays.min() - cur_norm).days)
            df.loc[idx, "days_until_holiday"] = min(days_until, 99)

        if len(past_holidays) > 0:
            days_after = int((cur_norm - past_holidays.max()).days)
            df.loc[idx, "days_after_holiday"] = min(days_after, 99)

    # 기존 v.6 스타일의 pre/post peak도 유지
    holiday_pd = pd.to_datetime(list(holiday_set))

    pre_holiday_map = {}
    for hd in holiday_pd:
        found_days = []
        offset = 1

        while len(found_days) < 2 and offset <= 7:
            prev_day = hd - pd.Timedelta(days=offset)

            if prev_day.weekday() not in [5, 6]:
                found_days.append(prev_day.normalize())

            offset += 1

        if len(found_days) >= 1:
            pre_holiday_map[found_days[0]] = 1
        if len(found_days) >= 2:
            if (found_days[0] - found_days[1]).days == 1:
                pre_holiday_map[found_days[1]] = 2

    post_holiday_map = {}
    for hd in holiday_pd:
        offset = 1

        while offset <= 7:
            next_day = hd + pd.Timedelta(days=offset)

            if next_day.weekday() not in [5, 6]:
                post_holiday_map[next_day.normalize()] = 1
                break

            offset += 1

    df["pre_holiday_peak"] = (
        df["날짜"]
        .map(pre_holiday_map)
        .fillna(0)
        .astype(int)
    )

    df["post_holiday_peak"] = (
        df["날짜"]
        .map(post_holiday_map)
        .fillna(0)
        .astype(int)
    )

    df.loc[df["holiday"] == 1, "pre_holiday_peak"] = 0
    df.loc[df["holiday"] == 1, "post_holiday_peak"] = 0

    return df


def add_seller_closed_features(df, seller_closed_dates=None):
    """
    셀러별 휴무일 feature 생성.
    미래 예측에서 이 값이 1이면 rule-based block 로직이 적용됨.
    """
    df = df.copy()

    seller_to_col = {
        "zigzag": "is_zigzag_closed",
        "kakao": "is_kakao_closed",
        "boosters": "is_boosters_closed",
        "other": "is_other_closed",
    }

    for col in seller_to_col.values():
        df[col] = 0

    if not seller_closed_dates:
        return df

    for seller, dates in seller_closed_dates.items():
        if seller not in seller_to_col:
            continue

        closed_dates = pd.to_datetime(dates).normalize()
        col = seller_to_col[seller]

        df.loc[df["날짜"].isin(closed_dates), col] = 1

    return df


def preprocess_features(
    df,
    manual_holidays=None,
    seller_closed_dates=None,
    manual_spikes_dict=None,
    event_periods=None,
):
    df = df.copy()
    df["날짜"] = pd.to_datetime(df["날짜"], errors="coerce").dt.normalize()
    df = df.sort_values("날짜").reset_index(drop=True)

    # ------------------------------------------------------
    # 순수_기타물량이 없으면 residual로 생성
    # ------------------------------------------------------
    needed_for_other = [TOTAL, ZIG_SAME, ZIG_NEXT, KAKAO, BOOSTERS]

    if OTHER_PURE not in df.columns and all(c in df.columns for c in needed_for_other):
        df[OTHER_PURE] = (
            pd.to_numeric(df[TOTAL], errors="coerce").fillna(0)
            - pd.to_numeric(df[ZIG_SAME], errors="coerce").fillna(0)
            - pd.to_numeric(df[ZIG_NEXT], errors="coerce").fillna(0)
            - pd.to_numeric(df[KAKAO], errors="coerce").fillna(0)
            - pd.to_numeric(df[BOOSTERS], errors="coerce").fillna(0)
        ).clip(lower=0)

    # ------------------------------------------------------
    # 1. 날짜 피처
    # ------------------------------------------------------
    df["month"] = df["날짜"].dt.month
    df["dow"] = df["날짜"].dt.weekday
    df["weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["week_of_year"] = df["날짜"].apply(lambda x: x.isocalendar().week)

    df["month"] = pd.Categorical(df["month"], categories=list(range(1, 13)))
    df["dow"] = pd.Categorical(df["dow"], categories=list(range(0, 7)))

    month_dummies = pd.get_dummies(df["month"], prefix="month").astype(int)
    dow_dummies = pd.get_dummies(df["dow"], prefix="dow").astype(int)

    df = pd.concat([df, month_dummies, dow_dummies], axis=1)

    df["month"] = df["month"].astype(int)
    df["dow"] = df["dow"].astype(int)

    # ------------------------------------------------------
    # 2. 이벤트 피처
    # ------------------------------------------------------
    if event_periods is None:
        event_periods = [
            ("2024-03-25", "2024-04-07"),
            ("2024-06-10", "2024-06-17"),
            ("2024-06-24", "2024-07-07"),
            ("2024-09-23", "2024-10-06"),
            ("2024-11-18", "2024-12-01"),
            ("2025-03-24", "2025-04-07"),
            ("2025-06-01", "2025-06-30"),
            ("2025-09-22", "2025-10-05"),
            ("2025-10-21", "2025-10-21"),
            ("2025-11-17", "2025-12-02"),
            ("2026-03-23", "2026-04-06"), # 직잿팟
            ("2026-05-18", "2026-05-31"), # 지그재그 11주년 행사 
        ]

    df["is_event"] = 0

    for start, end in event_periods:
        start_dt = pd.to_datetime(start)
        end_dt = pd.to_datetime(end)
        mask = (df["날짜"] >= start_dt) & (df["날짜"] <= end_dt)
        df.loc[mask, "is_event"] = 1

    # 이벤트 block/day index
    df["event_day_index"] = 0
    df["event_block_id"] = 0

    is_event = df["is_event"].eq(1)
    block_id = is_event.ne(is_event.shift()).cumsum()

    df.loc[is_event, "event_block_id"] = block_id[is_event].astype(int)
    df.loc[is_event, "event_day_index"] = (
        df[is_event]
        .groupby(block_id[is_event])
        .cumcount()
        + 1
    )

    df["event_after_1d"] = df["is_event"].shift(1).fillna(0).astype(int)
    df["event_after_2d"] = df["is_event"].shift(2).fillna(0).astype(int)
    df["event_before_1d"] = df["is_event"].shift(-1).fillna(0).astype(int)

    # ------------------------------------------------------
    # 3. 공휴일 피처
    # ------------------------------------------------------
    years = sorted(df["날짜"].dt.year.dropna().unique().tolist())
    holiday_set = build_holiday_set(years, manual_holidays=manual_holidays)

    # Timestamp vs date 타입 문제 방지: 반드시 .dt.date로 비교
    df["holiday"] = df["날짜"].dt.date.isin(holiday_set).astype(int)

    # 주말은 weekend로 따로 잡히므로 holiday에서는 제거
    df.loc[df["weekend"] == 1, "holiday"] = 0

    df = add_pre_post_holiday_features(df, holiday_set)

    # ------------------------------------------------------
    # 4. 셀러별 휴무 피처
    # ------------------------------------------------------
    df = add_seller_closed_features(
        df,
        seller_closed_dates=seller_closed_dates,
    )

    # ------------------------------------------------------
    # 5. 자동 spike 탐지
    # ------------------------------------------------------
    auto_cols = [c for c in ALL_TARGETS if c in df.columns]

    for col in auto_cols:
        df[f"{col}_spike_up_mild"] = 0
        df[f"{col}_spike_up_strong"] = 0
        df[f"{col}_spike_down"] = 0

        trend_col = f"{col}_trend"
        resid_col = f"{col}_resid_ratio"

        df[trend_col] = np.nan
        df[resid_col] = np.nan

        for d in range(7):
            idx_d = df.index[df["dow"] == d]
            if len(idx_d) == 0:
                continue

            s = pd.to_numeric(df.loc[idx_d, col], errors="coerce").sort_index()
            baseline = s.shift(1).rolling(window=3, min_periods=2).mean()
            resid = (s - baseline) / (baseline + 1e-5)

            df.loc[idx_d, trend_col] = baseline.astype(float).values
            df.loc[idx_d, resid_col] = resid.astype(float).values

        for wk_flag in [0, 1]:
            if wk_flag == 0:
                upper1, upper2, lower1 = 0.10, 0.20, -0.15
            else:
                upper1, upper2, lower1 = 0.20, 0.25, -0.10

            mask_up_mild = (
                (df[resid_col] >= upper1)
                & (df[resid_col] < upper2)
                & (df["weekend"] == wk_flag)
            )
            mask_up_strong = (
                (df[resid_col] >= upper2)
                & (df["weekend"] == wk_flag)
            )
            mask_down = (
                (df[resid_col] <= lower1)
                & (df["weekend"] == wk_flag)
            )

            df.loc[mask_up_mild, f"{col}_spike_up_mild"] = 1
            df.loc[mask_up_strong, f"{col}_spike_up_strong"] = 1
            df.loc[mask_down, f"{col}_spike_down"] = 1

    # ------------------------------------------------------
    # 6. 수동 spike 지정
    # ------------------------------------------------------
    if manual_spikes_dict:
        for tgt, spikes in manual_spikes_dict.items():
            for spike_type, dates in spikes.items():
                if dates:
                    dts = pd.to_datetime(dates).normalize()
                    spike_col = f"{tgt}_spike_{spike_type}"

                    if spike_col not in df.columns:
                        df[spike_col] = 0

                    df.loc[df["날짜"].isin(dts), spike_col] = 1

    # ------------------------------------------------------
    # 7. 기온 피처
    # ------------------------------------------------------
    if "avg_temp" in df.columns:
        df = add_temp_shift_features(df, temp_col="avg_temp")
        df = df.drop(columns=["avg_temp"], errors="ignore")

    # ------------------------------------------------------
    # 8. 각 타겟 lag/rolling 생성
    # ------------------------------------------------------
    for tgt in ALL_TARGETS:
        df = add_target_lag_rolling_features(
            df,
            target_col=tgt,
            lags=(1, 7, 14, 21, 28),
            roll_windows=(7, 14, 21, 28),
        )

    # trend/resid는 spike 탐지용이므로 학습에서는 제거
    drop_cols = []
    for col in auto_cols:
        drop_cols += [f"{col}_trend", f"{col}_resid_ratio"]

    df = df.drop(columns=drop_cols, errors="ignore")
    return df


# ==========================================================
# 2. XGBoost Helper
# ==========================================================
def weighted_mse_alpha(alpha=1.5):
    """
    과소예측을 더 강하게 벌점 주는 custom objective.
    label - pred > 0이면 실제가 예측보다 큰 것, 즉 과소예측.
    """
    def obj(label, pred):
        residual = label - pred
        grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
        hess = np.where(residual > 0, 2 * alpha, 2.0)
        return grad, hess

    return obj


def _ensure_cols(df, cols, fill_value=0.0):
    out = df.copy()
    missing = [c for c in cols if c not in out.columns]

    for c in missing:
        out[c] = fill_value

    return out[cols]


def _valid_target_mask(df, target_col, zero_run_min=3):
    """
    오래 이어지는 0은 실제 휴무/데이터 결측/운영 중단 가능성이 있으므로
    학습에서 제외하기 위한 mask.
    """
    s = pd.to_numeric(df[target_col], errors="coerce")
    base_mask = s.notna() & (s > 0)

    is_zero = s.fillna(0).eq(0)
    group_id = is_zero.ne(is_zero.shift()).cumsum()
    zero_run_len = is_zero.groupby(group_id).transform("sum")

    long_zero_mask = is_zero & (zero_run_len >= zero_run_min)
    return base_mask & (~long_zero_mask)


def _clean_lag_features_from_fake_zero(df, target_cols, zero_run_min=3):
    """
    긴 0 구간이 lag/rolling에 그대로 들어가면 미래 예측을 과하게 낮출 수 있으므로
    feature 생성 전 NaN으로 바꿔 rolling 계산에서 영향 완화.
    """
    out = df.copy()

    for col in target_cols:
        if col not in out.columns:
            continue

        s = pd.to_numeric(out[col], errors="coerce")
        is_zero = s.fillna(0).eq(0)
        group_id = is_zero.ne(is_zero.shift()).cumsum()
        zero_run_len = is_zero.groupby(group_id).transform("sum")

        fake_zero_mask = is_zero & (zero_run_len >= zero_run_min)
        out.loc[fake_zero_mask, col] = np.nan

    return out


def _overwrite_ts_features(row_df, tgt, buf):
    """
    Recursive forecasting에서 미래 row의 lag/rolling을 최신 buffer로 덮어씀.
    """
    row = row_df.copy()

    clean_buf = [
        float(x)
        for x in buf
        if pd.notna(x) and float(x) > 0
    ]

    for lag in (1, 7, 14, 21, 28):
        row[f"{tgt}_lag_{lag}"] = clean_buf[-lag] if len(clean_buf) >= lag else 0.0

    for w in (7, 14, 21, 28):
        tail = clean_buf[-w:]

        row[f"{tgt}_roll_mean_{w}"] = float(np.mean(tail)) if len(tail) >= 2 else 0.0
        row[f"{tgt}_roll_std_{w}"] = float(np.std(tail, ddof=1)) if len(tail) >= 2 else 0.0

    return row


def apply_seller_block_postprocess(row, tgt, pred_i, other_closed_ratio=0.5):
    """
    셀러별 휴무일 강제 block.
    - 지그재그 휴무: 지그재그_당일/익일 = 0
    - 카카오 휴무: 카카오모빌리티 = 0
    - 부스터스 휴무: 부스터스 = 0
    - 기타 휴무: 순수_기타물량은 비율 감소 옵션 적용
    """
    pred_i = float(pred_i)

    if tgt in [ZIG_SAME, ZIG_NEXT]:
        if int(row.get("is_zigzag_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == KAKAO:
        if int(row.get("is_kakao_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == BOOSTERS:
        if int(row.get("is_boosters_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == OTHER_PURE:
        if int(row.get("is_other_closed", pd.Series([0])).iloc[0]) == 1:
            return max(0.0, pred_i * other_closed_ratio)

    return max(0.0, pred_i)


def build_feature_dict(full_df):
    """
    Target leakage 방지용 feature set 구성.
    모든 raw target, 당일 집계 컬럼, 로깅용 실적 컬럼 제거.
    단, 해당 target의 lag/rolling/spike는 사용 가능.
    """
    all_cols = full_df.columns.tolist()
    features_dict = {}

    leakage_cols = [
        "날짜",
        TOTAL,
        ZIG_SAME,
        ZIG_NEXT,
        KAKAO,
        BOOSTERS,
        OTHER_PURE,

        # 기존/로깅/당일 집계 컬럼들
        "지그재그 당일배송",
        "other_seller",
        "기타_당일",
        "기타_익일",
        "기타_물량",
        "기타_총물량",
        "당일_물량",
        "익일_물량",
        "셀러수",
    ]

    for tgt in BASE_TARGETS:
        exclude_other_target_features = []

        for other in ALL_TARGETS:
            if other == tgt:
                continue

            exclude_other_target_features += [
                c for c in all_cols
                if c.startswith(f"{other}_lag_")
                or c.startswith(f"{other}_roll_")
                or c.startswith(f"{other}_spike_")
            ]

        features_dict[tgt] = [
            c for c in all_cols
            if c not in leakage_cols + exclude_other_target_features
        ]

    # Meta 모델은 raw target은 금지.
    # 대신 recursive로 만들 수 있는 TOTAL lag/rolling, 일반 feature, base 예측값은 사용.
    meta_leakage_cols = [
        "날짜",
        TOTAL,
        ZIG_SAME,
        ZIG_NEXT,
        KAKAO,
        BOOSTERS,
        OTHER_PURE,
        "지그재그 당일배송",
        "other_seller",
        "기타_당일",
        "기타_익일",
        "기타_물량",
        "기타_총물량",
        "당일_물량",
        "익일_물량",
        "셀러수",
    ]

    # Meta에서 base target의 lag/rolling은 제외하고, TOTAL lag/rolling만 남김
    exclude_base_ts_for_meta = []
    for base_tgt in BASE_TARGETS:
        exclude_base_ts_for_meta += [
            c for c in all_cols
            if c.startswith(f"{base_tgt}_lag_")
            or c.startswith(f"{base_tgt}_roll_")
            or c.startswith(f"{base_tgt}_spike_")
        ]

    features_dict[TOTAL] = [
        c for c in all_cols
        if c not in meta_leakage_cols + exclude_base_ts_for_meta
    ]

    return features_dict


# ==========================================================
# 3. 학습/검증/미래 예측 함수
# ==========================================================
def train_validate_forecast(
    train,
    forecast_days=7,
    valid_days=7,
    temp_future_df=None,
    manual_holidays=None,
    seller_closed_dates=None,
    manual_spikes_train=None,
    manual_spikes_future=None,
    manual_seller_fixed_volumes=None,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=True,
):
    RANDOM_STATE = 42
    MIN_TRAIN_SAMPLES = 30
    ZERO_RUN_MIN = 3

    train_sorted = train.copy()
    train_sorted["날짜"] = pd.to_datetime(train_sorted["날짜"], errors="coerce").dt.normalize()
    train_sorted = train_sorted.sort_values("날짜").reset_index(drop=True)

    # 순수 기타 타겟 재계산
    if all(c in train_sorted.columns for c in [TOTAL, ZIG_SAME, ZIG_NEXT, KAKAO, BOOSTERS]):
        train_sorted[OTHER_PURE] = (
            pd.to_numeric(train_sorted[TOTAL], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[ZIG_SAME], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[ZIG_NEXT], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[KAKAO], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[BOOSTERS], errors="coerce").fillna(0)
        ).clip(lower=0)

    train_for_feature = _clean_lag_features_from_fake_zero(
        train_sorted,
        target_cols=BASE_TARGETS,
        zero_run_min=ZERO_RUN_MIN,
    )

    train_feat_all = preprocess_features(
        train_for_feature,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_dict=manual_spikes_train,
    )

    split_idx = len(train_feat_all) - valid_days
    train_df = train_feat_all.iloc[:split_idx].copy()
    valid_df = train_feat_all.iloc[split_idx:].copy()
    full_df = train_feat_all.copy()

    last_date = train_sorted["날짜"].max()
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=forecast_days,
    )

    future_df = pd.DataFrame({"날짜": future_dates})

    if temp_future_df is not None:
        temp_future_df = temp_future_df.copy()
        temp_future_df["날짜"] = pd.to_datetime(temp_future_df["날짜"], errors="coerce").dt.normalize()

        future_temp_input = pd.merge(
            future_df,
            temp_future_df[["날짜", "avg_temp"]],
            on="날짜",
            how="left",
        )

        # 예보가 비는 날짜는 앞뒤 값 또는 15도로 대체
        future_temp_input["avg_temp"] = (
            future_temp_input["avg_temp"]
            .ffill()
            .bfill()
            .fillna(15.0)
        )
    else:
        future_temp_input = future_df.copy()
        future_temp_input["avg_temp"] = 15.0

    future_feat_df = preprocess_features(
        future_temp_input,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_dict=manual_spikes_future,
    )

    future_feat_df = future_feat_df.iloc[-forecast_days:].reset_index(drop=True)

    for tgt in ALL_TARGETS:
        if tgt not in future_feat_df.columns:
            future_feat_df[tgt] = np.nan

    if debug_future_features:
        print("\n=== 미래 feature 체크 ===")
        debug_cols = [
            "날짜",
            "dow",
            "weekend",
            "holiday",
            "pre_holiday_peak",
            "post_holiday_peak",
            "days_until_holiday",
            "days_after_holiday",
            "is_zigzag_closed",
            "is_kakao_closed",
            "is_boosters_closed",
            "is_other_closed",
        ]
        display(future_feat_df[[c for c in debug_cols if c in future_feat_df.columns]])

    features_dict = build_feature_dict(full_df)

    xgb_base_params = {
        "n_estimators": 900,
        "learning_rate": 0.015,
        "max_depth": 3,
        "min_child_weight": 5,
        "gamma": 0.05,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.1,
        "reg_lambda": 3.0,
        "objective": weighted_mse_alpha(alpha=1.6),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    xgb_meta_params = {
        "n_estimators": 500,
        "learning_rate": 0.02,
        "max_depth": 2,
        "min_child_weight": 8,
        "gamma": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.2,
        "reg_lambda": 5.0,
        "objective": weighted_mse_alpha(alpha=1.3),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    trained_models = {}
    pred_train_base = {}
    pred_valid_base = {}
    pred_full_base = {}

    print("\n=== Base 모델 학습 제외 데이터 확인 ===")

    for tgt in BASE_TARGETS:
        train_recent = train_df.tail(base_recent_window).copy()
        full_recent = full_df.tail(base_recent_window).copy()

        X_tr = _ensure_cols(train_recent, features_dict[tgt]).fillna(0.0)
        y_tr = pd.to_numeric(train_recent[tgt], errors="coerce")

        X_val = _ensure_cols(valid_df, features_dict[tgt]).fillna(0.0)

        X_train_all = _ensure_cols(train_df, features_dict[tgt]).fillna(0.0)
        X_full_all = _ensure_cols(full_df, features_dict[tgt]).fillna(0.0)

        X_full_recent = _ensure_cols(full_recent, features_dict[tgt]).fillna(0.0)
        y_full_recent = pd.to_numeric(full_recent[tgt], errors="coerce")

        mask_tr = _valid_target_mask(train_recent, tgt, ZERO_RUN_MIN)
        mask_full = _valid_target_mask(full_recent, tgt, ZERO_RUN_MIN)

        print(
            f"{tgt}: train 최근 {len(train_recent)}건 중 사용 {mask_tr.sum()}건 / 제외 {len(mask_tr) - mask_tr.sum()}건, "
            f"full 최근 {len(full_recent)}건 중 사용 {mask_full.sum()}건 / 제외 {len(mask_full) - mask_full.sum()}건"
        )

        if mask_tr.sum() >= MIN_TRAIN_SAMPLES:
            model = XGBRegressor(**xgb_base_params)
            model.fit(
                X_tr.loc[mask_tr],
                np.log1p(y_tr.loc[mask_tr].astype(float)),
                verbose=False,
            )

            pred_train_base[tgt] = np.clip(np.expm1(model.predict(X_train_all)), 0, None)
            pred_valid_base[tgt] = np.clip(np.expm1(model.predict(X_val)), 0, None)

            model_full = XGBRegressor(**xgb_base_params)
            model_full.fit(
                X_full_recent.loc[mask_full],
                np.log1p(y_full_recent.loc[mask_full].astype(float)),
                verbose=False,
            )

            pred_full_base[tgt] = np.clip(np.expm1(model_full.predict(X_full_all)), 0, None)
            trained_models[tgt] = model_full

        else:
            recent_valid = y_full_recent.loc[mask_full].tail(28)
            mean_val = float(recent_valid.mean()) if len(recent_valid) > 0 else 0.0

            pred_train_base[tgt] = np.full(len(X_train_all), mean_val)
            pred_valid_base[tgt] = np.full(len(X_val), mean_val)
            pred_full_base[tgt] = np.full(len(X_full_all), mean_val)
            trained_models[tgt] = None

    print("====================================\n")

    # ------------------------------------------------------
    # Meta 모델 학습
    # ------------------------------------------------------
    meta_features = features_dict[TOTAL] + [f"pred_{t}" for t in BASE_TARGETS] + ["base_sum_pred"]

    X_tr_meta = _ensure_cols(train_df, features_dict[TOTAL]).fillna(0.0)
    X_val_meta = _ensure_cols(valid_df, features_dict[TOTAL]).fillna(0.0)
    X_full_meta = _ensure_cols(full_df, features_dict[TOTAL]).fillna(0.0)

    for tgt in BASE_TARGETS:
        X_tr_meta[f"pred_{tgt}"] = pred_train_base[tgt]
        X_val_meta[f"pred_{tgt}"] = pred_valid_base[tgt]
        X_full_meta[f"pred_{tgt}"] = pred_full_base[tgt]

    X_tr_meta["base_sum_pred"] = sum(X_tr_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_val_meta["base_sum_pred"] = sum(X_val_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_full_meta["base_sum_pred"] = sum(X_full_meta[f"pred_{t}"] for t in BASE_TARGETS)

    X_tr_meta = X_tr_meta[meta_features]
    X_val_meta = X_val_meta[meta_features]
    X_full_meta = X_full_meta[meta_features]

    y_tr_meta = pd.to_numeric(train_df[TOTAL], errors="coerce")
    y_full_meta = pd.to_numeric(full_df[TOTAL], errors="coerce")

    train_meta_recent_idx = train_df.tail(meta_recent_window).index
    full_meta_recent_idx = full_df.tail(meta_recent_window).index

    X_tr_meta_recent = X_tr_meta.loc[train_meta_recent_idx]
    y_tr_meta_recent = y_tr_meta.loc[train_meta_recent_idx]
    train_df_meta_recent = train_df.loc[train_meta_recent_idx]

    X_full_meta_recent = X_full_meta.loc[full_meta_recent_idx]
    y_full_meta_recent = y_full_meta.loc[full_meta_recent_idx]
    full_df_meta_recent = full_df.loc[full_meta_recent_idx]

    mask_tr_meta = _valid_target_mask(train_df_meta_recent, TOTAL, ZERO_RUN_MIN)
    mask_full_meta = _valid_target_mask(full_df_meta_recent, TOTAL, ZERO_RUN_MIN)

    print(
        f"{TOTAL}: meta 최근 {len(train_df_meta_recent)}건 중 사용 {mask_tr_meta.sum()}건 / 제외 {len(mask_tr_meta) - mask_tr_meta.sum()}건"
    )

    model_meta = XGBRegressor(**xgb_meta_params)
    model_meta.fit(
        X_tr_meta_recent.loc[mask_tr_meta],
        np.log1p(y_tr_meta_recent.loc[mask_tr_meta].astype(float)),
        verbose=False,
    )

    meta_valid_pred = np.clip(np.expm1(model_meta.predict(X_val_meta)), 0, None)
    base_sum_valid_pred = X_val_meta["base_sum_pred"].values

    pred_valid_total = (
        (1 - meta_blend_weight) * base_sum_valid_pred
        + meta_blend_weight * meta_valid_pred
    )

    model_meta_full = XGBRegressor(**xgb_meta_params)
    model_meta_full.fit(
        X_full_meta_recent.loc[mask_full_meta],
        np.log1p(y_full_meta_recent.loc[mask_full_meta].astype(float)),
        verbose=False,
    )

    trained_models[TOTAL] = model_meta_full

    # ------------------------------------------------------
    # 검증 성능 출력
    # ------------------------------------------------------
    print("\n=== 검증 성능 (MAPE) ===")

    actual_total = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    mask_total = actual_total > 0

    if mask_total.sum() > 0:
        mape_total = mean_absolute_percentage_error(
            actual_total[mask_total],
            pred_valid_total[mask_total],
        )
        print(f"총 물량: {mape_total:.4f} (정확도 {100 * (1 - mape_total):.2f}%)")

    for tgt in BASE_TARGETS:
        actuals = pd.to_numeric(valid_df[tgt], errors="coerce").values
        preds = pred_valid_base[tgt]
        mask = actuals > 0

        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(actuals[mask], preds[mask])
            print(f"{tgt}: {mape:.4f} (정확도 {100 * (1 - mape):.2f}%)")

    print("========================\n")

    # ------------------------------------------------------
    # 미래 Recursive Forecasting
    # ------------------------------------------------------
    bufs = {
        tgt: pd.to_numeric(full_df[tgt], errors="coerce").tolist()
        for tgt in ALL_TARGETS
    }

    future_preds = {tgt: [] for tgt in ALL_TARGETS}

    for i in range(forecast_days):
        row = future_feat_df.iloc[[i]].copy()
        cur_date_str = row["날짜"].dt.strftime("%Y-%m-%d").iloc[0] # 현재 날짜 문자열 추출

        for tgt in BASE_TARGETS:
            row = _overwrite_ts_features(row, tgt, bufs[tgt])
            X_row = _ensure_cols(row, features_dict[tgt]).fillna(0.0)

            if trained_models[tgt] is not None:
                pred_i = float(np.expm1(trained_models[tgt].predict(X_row))[0])
                pred_i = max(0.0, pred_i)
            else:
                recent_buf = [
                    x
                    for x in bufs[tgt]
                    if pd.notna(x) and float(x) > 0
                ]
                pred_i = float(np.mean(recent_buf[-7:])) if len(recent_buf) > 0 else 0.0

            # 핵심: 셀러별 휴무일 block 후처리
            pred_i = apply_seller_block_postprocess(
                row=row,
                tgt=tgt,
                pred_i=pred_i,
                other_closed_ratio=other_closed_ratio,
            )

            # 확정된 물량은 덮어쓰기
            if manual_seller_fixed_volumes and tgt in manual_seller_fixed_volumes:
                if cur_date_str in manual_seller_fixed_volumes[tgt]:
                    pred_i = float(manual_seller_fixed_volumes[tgt][cur_date_str])

            future_preds[tgt].append(pred_i)
            bufs[tgt].append(pred_i)
            row[f"pred_{tgt}"] = pred_i

        base_sum_pred = sum(row[f"pred_{t}"].iloc[0] for t in BASE_TARGETS)
        row["base_sum_pred"] = base_sum_pred

        row = _overwrite_ts_features(row, TOTAL, bufs[TOTAL])
        X_row_meta = _ensure_cols(row, meta_features).fillna(0.0)

        meta_pred = max(
            0.0,
            float(np.expm1(trained_models[TOTAL].predict(X_row_meta))[0]),
        )

        pred_total = (
            (1 - meta_blend_weight) * base_sum_pred
            + meta_blend_weight * meta_pred
        )

        future_preds[TOTAL].append(pred_total)
        bufs[TOTAL].append(pred_total)

    # ------------------------------------------------------
    # 결과 테이블 구성
    # ------------------------------------------------------
    valid_result_df = pd.DataFrame({
        "날짜": valid_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    valid_result_df["실제_총물량"] = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    valid_result_df["예측_총물량"] = np.round(pred_valid_total).astype(int)
    valid_result_df["base_sum_예측"] = np.round(base_sum_valid_pred).astype(int)
    valid_result_df["meta_예측"] = np.round(meta_valid_pred).astype(int)

    for tgt in BASE_TARGETS:
        valid_result_df[f"실제_{tgt}"] = pd.to_numeric(valid_df[tgt], errors="coerce").values
        valid_result_df[f"예측_{tgt}"] = np.round(pred_valid_base[tgt]).astype(int)

    forecast_df = pd.DataFrame({
        "날짜": future_feat_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    for tgt in BASE_TARGETS:
        forecast_df[tgt] = np.round(future_preds[tgt]).astype(int)

    forecast_df["base_sum"] = forecast_df[BASE_TARGETS].sum(axis=1)
    forecast_df["총 물량"] = np.round(future_preds[TOTAL]).astype(int)

    # 휴무일 feature도 결과에 같이 표시해서 block 확인 가능하게 함
    closed_cols = [
        "is_zigzag_closed",
        "is_kakao_closed",
        "is_boosters_closed",
        "is_other_closed",
        "holiday",
        "pre_holiday_peak",
        "post_holiday_peak",
    ]

    for col in closed_cols:
        if col in future_feat_df.columns:
            forecast_df[col] = future_feat_df[col].values

    return trained_models, forecast_df, valid_result_df, features_dict

### Prediction

휴일 및 spike 설정

In [ ]:
# ==========================================================
# Cell 3. 실행 코드
# ==========================================================

# 날짜 형식 정리
data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"], errors="coerce").dt.normalize()
data["avg_temp"] = pd.to_numeric(data["avg_temp"], errors="coerce")
temp_merge_data["avg_temp"] = pd.to_numeric(temp_merge_data["avg_temp"], errors="coerce")

# ----------------------------------------------------------
# 1. 수동 공휴일
# ----------------------------------------------------------
manual_holidays = [
    "2026-05-01",  # 근로자의 날
]

# ----------------------------------------------------------
# 2. 셀러별 휴무일
# ----------------------------------------------------------
seller_closed_dates = {
    "zigzag": [],   # 지그재그 당일 배송 휴무일
    "kakao": ["2026-05-05", "2026-05-25"], # 카카오 모빌리티 휴무일
    "boosters": ["2026-05-01","2026-05-02","2026-05-05", "2026-05-25"], # 부스터스 휴무일
    "other": ["2026-05-01","2026-05-02","2026-05-05", "2026-05-25"]   # 그 외 셀러들 휴무일(과반 이상 휴무면, 휴무로 태그 넣기)
}

# ----------------------------------------------------------
# 3. 수동 spike
# ----------------------------------------------------------
manual_spikes_future = {
    TOTAL: {
        "down": [],
    },
    ZIG_SAME: {
        "down": [],
    },
    ZIG_NEXT: {
        "down": [],
    },
    KAKAO: {
        "down": [],
    },
    BOOSTERS: {
        "down": [],
    },
    OTHER_PURE: {
        "down": [],
    },
}

# 학습 데이터에는 특별히 강제할 spike가 없으면 None
manual_spikes_train = None


지그재그에서 행사 제공치 제공해줄 경우 해당 물량 입력 후 최종 예측 실행

In [ ]:
# ----------------------------------------------------------
# 4. 화주사 제공 확정 물량 셋팅
# ----------------------------------------------------------
# 평상시: 아래처럼 딕셔너리를 비워두거나 값을 주석 처리합니다.
# 행사시: 날짜와 물량을 입력합니다.
manual_seller_fixed_volumes = {
    # "지그재그_당일": {"2026-05-20": 4500, "2026-05-21": 5200},
    # "지그재그_익일": {"2026-05-20": 1200, "2026-05-21": 1500}
    "지그재그_당일": {"2026-05-20": 7019, "2026-05-21": 7602, "2026-05-22": 5647, "2026-05-23": 4744, "2026-05-24": 5361, "2026-05-25": 5464, "2026-05-26": 5556},
    "지그재그_익일": {"2026-05-20": 12936, "2026-05-21": 14334, "2026-05-22": 11709, "2026-05-23": 9510, "2026-05-24": 10002, "2026-05-25": 10437, "2026-05-26": 10638}

}

# ----------------------------------------------------------
# 5. 모델 실행 (기본 순수 예측 1회)
# ----------------------------------------------------------
print("⏳ 기본 모델 예측을 진행 중입니다...")
trained_models, forecast_df_pure, valid_result, features_dict = train_validate_forecast(
    train=data,
    forecast_days=7,
    valid_days=7,
    temp_future_df=temp_merge_data,
    manual_holidays=manual_holidays,
    seller_closed_dates=seller_closed_dates,
    manual_spikes_train=manual_spikes_train,
    manual_spikes_future=manual_spikes_future,
    manual_seller_fixed_volumes=None, # 기본 예측이므로 None
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=False,
)

# 최종 사용할 결과를 일단 기본 예측값으로 맵핑
forecast_df_final = forecast_df_pure.copy()

# ----------------------------------------------------------
# 6. 이벤트 시: 고정 물량 반영 및 Before/After 비교 리포트 산출
# ----------------------------------------------------------
if manual_seller_fixed_volumes: # 딕셔너리에 값이 존재할 경우에만 작동
    print("\n⏳ 확정 물량 고정 반영 모델을 추가 연산합니다...")
    _, forecast_df_fixed, _, _ = train_validate_forecast(
        train=data,
        forecast_days=7,
        valid_days=7,
        temp_future_df=temp_merge_data,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_train=manual_spikes_train,
        manual_spikes_future=manual_spikes_future,
        manual_seller_fixed_volumes=manual_seller_fixed_volumes, # 확정 물량 주입
        base_recent_window=150,
        meta_recent_window=240,
        meta_blend_weight=0.2,
        other_closed_ratio=0.5,
        debug_future_features=False,
    )
    
    # 덮어쓰기 연산이 끝났으므로 최종 리포트 파일은 이걸로 저장하도록 스위치
    forecast_df_final = forecast_df_fixed 
    
    # 비교 DF 생성
    comparison_df = forecast_df_fixed[['날짜', '총 물량', '지그재그_당일', '지그재그_익일']].rename(columns={
        '총 물량': '총물량(고정)', '지그재그_당일': '지그재그_당일(고정)', '지그재그_익일': '지그재그_익일(고정)'
    })
    comparison_df['총물량(순수)'] = forecast_df_pure['총 물량']
    comparison_df['지그재그_당일(순수)'] = forecast_df_pure['지그재그_당일']
    comparison_df['지그재그_익일(순수)'] = forecast_df_pure['지그재그_익일']
    comparison_df['총물량_Delta'] = comparison_df['총물량(고정)'] - comparison_df['총물량(순수)']
    
    comparison_df = comparison_df[[
        '날짜', '지그재그_당일(순수)', '지그재그_당일(고정)', 
        '지그재그_익일(순수)', '지그재그_익일(고정)', 
        '총물량(순수)', '총물량(고정)', '총물량_Delta'
    ]]
    print("\n📊 [행사 확정 물량 반영 Before / After 비교 리포트]")
    display(comparison_df)

else:
    print("\n✅ 입력된 확정 물량(이벤트)이 없어 일반 예측 모드로 종료되었습니다.")

# 결과물은 forecast_df_final을 사용해 저장
display(forecast_df_final)

# 지역별 물량 예측

In [17]:
# 최신 데이터 로딩 파이프라인
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
import os
from dotenv import load_dotenv

class AutoContainerGeneration:
    def __init__(self, version=1, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

        load_dotenv()

    # 공통 MySQL 데이터 추출 메서드
    def fetch_data(self, query, ssh_host, ssh_user, ssh_private_key,
                   mysql_host, mysql_port, mysql_user, mysql_password, mysql_database):
        """
        MySQL 데이터를 추출하여 DF로 변환
        """
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port)
            ) as tunnel:
                print("SSH 터널 연결 성공")

                with pymysql.connect(
                    host='127.0.0.1',
                    user=mysql_user,
                    passwd=mysql_password,
                    db=mysql_database,
                    charset='utf8',
                    port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")

                        df = pd.DataFrame(results)
                        return df

        except Exception as e:
            print(f"Error fetching data for: {e}")
            return None

    # 물량 집계 데이터 추출
    def fetch_cluster_data(self):

        # SSH 및 MySQL 설정
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv(r"SSH_PRIVATE_KEY")

        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        print(f'Environment: {mysql_host}')

        # 실행할 쿼리
        cluster_query = """
                SELECT
                    hub_cluster_plan.plan_date AS 날짜,
                    shipping_shippingitem.id AS 주문번호,
                    shop_shop.name AS 셀러명,
                    CASE 
                        WHEN RIGHT(SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1), 1)
                            IN ('가','나','다','라','마','바','사','아','자','차','카','타','파','하')
                        THEN CONCAT(
                                SUBSTRING_INDEX(ls.name, ' ', 1), ' ',
                                IF(SUBSTRING_INDEX(ls.name, ' ', 1) = '서울특별시',
                                CONCAT(
                                    LEFT(
                                        SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1),
                                        CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1)) - 1
                                    ), '구'
                                ),
                                CONCAT(
                                    LEFT(
                                        SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1),
                                        CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1)) - 1
                                    ), '시'
                                )
                                )
                            )
                        ELSE ls.name
                    END AS 지역
                FROM shipping_shippingitem
                JOIN order_order
                    ON shipping_shippingitem.order_id = order_order.id
                JOIN hub_cluster_plan_item
                    ON shipping_shippingitem.id = hub_cluster_plan_item.shipping_item_id
                JOIN hub_cluster_plan
                    ON hub_cluster_plan_item.cluster_plan_id = hub_cluster_plan.id
                JOIN location_address la
                    ON shipping_shippingitem.address_id = la.id
                JOIN location_sector ls
                    ON shipping_shippingitem.designated_sector_id = ls.id
                JOIN shop_shop
                    ON shop_shop.id = order_order.shop_id
                WHERE hub_cluster_plan.plan_date <= CURRENT_DATE()
                AND hub_cluster_plan.plan_no = 1
                AND ls.allowed = 1
        """

        # 쿼리 실행
        df_forecast = self.fetch_data(
            cluster_query,
            ssh_host, ssh_user, ssh_private_key,
            mysql_host, mysql_port, mysql_user, mysql_password, mysql_database
        )

        return df_forecast

# DaaS DB 데이터 추출
if __name__ == "__main__":
    generator = AutoContainerGeneration(version=1, debug=True)
    df_forecast = generator.fetch_cluster_data()

    # 각 DataFrame 확인
    print("최신 DF 생성 완료")

# 기존 데이터 업로드
base = pd.read_excel("./data/region_latest_cl.xlsx")
print("기존 데이터 업로드 완료")

# 최신 데이터 전처리
df_forecast.loc[df_forecast['지역']=="경기도 남양주", "지역"] = "경기도 남양주시"
df_forecast.loc[df_forecast['지역']=="경기도 성남시 수정가", "지역"] = "경기도 성남시 수정구"

total = df_forecast.groupby(['날짜','지역'])['주문번호'].agg({('총 물량','count')}).reset_index()
zigzag = df_forecast[df_forecast['셀러명']=="지그재그 당일배송"].groupby(['날짜','지역'])['주문번호'].agg({('지그재그 당일배송','count')}).reset_index()
kakao = df_forecast[df_forecast['셀러명']=="카카오모빌리티"].groupby(['날짜','지역'])['주문번호'].agg({('카카오모빌리티','count')}).reset_index()
bosters = df_forecast[df_forecast['셀러명']=="부스터스"].groupby(['날짜','지역'])['주문번호'].agg({('부스터스','count')}).reset_index()

total = pd.merge(total, zigzag, how = 'left', on = ['날짜','지역'])
total = pd.merge(total, kakao, how = 'left', on = ['날짜','지역'])
total = pd.merge(total, bosters, how = 'left', on = ['날짜','지역'])

merge_data = total.groupby(['날짜','지역'])[['총 물량','지그재그 당일배송','카카오모빌리티','부스터스']].sum().reset_index()
merge_data['카카오모빌리티'] = merge_data['카카오모빌리티'].astype(int)
merge_data['부스터스'] = merge_data['부스터스'].astype(int)

## 날짜 오름차순, 지역 오름차순 정렬
merge_data = merge_data.sort_values(by=['지역', '날짜']).reset_index(drop=True)

# 데이터 업데이트
data = pd.concat([base, merge_data], axis=0).reset_index(drop=True)
data['날짜'] = pd.to_datetime(data['날짜'], format='%Y-%m-%d', errors='coerce')
print("데이터 업데이트 완료")

# 중복 제거 및 날짜 & 지역 정렬
data.loc[data['지역']=="인천광역시 계양시", "지역"] = "인천광역시 계양구"
data.loc[data['지역']=="인천광역시 서구시", "지역"] = "인천광역시 서구"
data = data.drop_duplicates(subset=['날짜','지역'], keep='first').reset_index(drop=True)
data = data.sort_values(by=['지역', '날짜']).reset_index(drop=True)

## other_seller 정의
data['other_seller'] = data['총 물량'] - (data['지그재그 당일배송'] + data['카카오모빌리티'] + data['부스터스'])
print("데이터 중복 제거")

# 데이터 오류 정정
## 서울특별시 금천구 물량 조정
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "총 물량"] = 246
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "지그재그 당일배송"] = 200
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "카카오모빌리티"] = 37
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "부스터스"] = 5
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "other_seller"] = 4
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "총 물량"] = 139
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "지그재그 당일배송"] = 126
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "카카오모빌리티"] = 13
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "부스터스"] = 0
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "other_seller"] = 0

# data.loc[data['날짜']=="2025-12-05","총 물량"] = (data.loc[data['날짜']=="2025-12-05","총 물량"]*0.92).astype(int)

## 2023-01-19일자 데이터 삭제
data = data[data['날짜'] != "2023-01-19"].sort_values(by=['지역', '날짜']).reset_index(drop=True)

# 신규 데이터 저장
data.to_excel("./data/region_latest_cl.xlsx", index=False)
print("신규 데이터 저장 완료")

print("학습 데이터 구축 완료!")
print("-"*70)

# ----------------------------------------------------------------------------------------------------------
# 기온 데이터 API 호출 및 병합

import requests
import pandas as pd
import numpy as np
import urllib3
from tqdm import tqdm
from datetime import datetime, timedelta

# -----------------------
# 0. 공통 설정
# -----------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

serviceKey = "ZjfdmTgpRQm33Zk4KZUJIQ"

# ===== 단기예보 =====
def fetch_short_term_forecast(base_date: str, base_time: str, region_point: dict):
    """단기예보 API 호출 → 날짜별 평균기온 DataFrame 반환"""
    records_all = []

    for region, (nx, ny) in tqdm(region_point.items(), desc="단기예보 수집"):
        url = (
            f"https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getVilageFcst"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&base_date={base_date}&base_time={base_time}"
            f"&nx={nx}&ny={ny}&authKey={serviceKey}"
        )
        try:
            response = requests.get(url, verify=False, timeout=20)
            response.raise_for_status()
            json_response = response.json()
            items = json_response["response"]["body"]["items"]["item"]
        except Exception as e:
            print(f"⚠️ {region} 단기 데이터 요청 실패: {e}")
            continue

        informations = {}
        for it in items:
            cate = it["category"]
            fcstDate = it["fcstDate"]
            fcstTime = it["fcstTime"]
            fcstValue = it["fcstValue"]
            key = f"{fcstDate}_{fcstTime}"
            if key not in informations:
                informations[key] = {}
            informations[key][cate] = fcstValue

        region_records = []
        for key, vals in informations.items():
            if "TMP" in vals:
                date = key.split("_")[0]
                region_records.append({"date": date, "TMP": float(vals["TMP"])})

        if not region_records:
            continue

        df_region = (
            pd.DataFrame(region_records)
            .groupby("date", as_index=False)["TMP"]
            .mean()
            .rename(columns={"TMP": "avg_temp"})
        )
        df_region["지역"] = region
        records_all.append(df_region)

    if not records_all:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    df_all = pd.concat(records_all, ignore_index=True)
    df_all.rename(columns={"date": "날짜"}, inplace=True)

    # 날짜 형식 통일
    df_all["날짜"] = pd.to_datetime(df_all["날짜"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    return df_all

# ===== 중기예보 =====
def fetch_mid_term_forecast(base_date: str, region_code: dict):
    """중기예보 API 호출 → 날짜별 평균기온 DataFrame 반환"""
    records = []

    for region, regId in tqdm(region_code.items(), desc="중기예보 수집"):
        url = (
            f"https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService/getMidTa"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&regId={regId}&tmFc={base_date}0600&authKey={serviceKey}"
        )
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            data = response.json()["response"]["body"]["items"]["item"][0]
        except Exception as e:
            print(f"⚠️ {region}({regId}) 중기 데이터 요청 실패: {e}")
            continue

        for day in range(4, 11):
            ta_min = data.get(f"taMin{day}")
            ta_max = data.get(f"taMax{day}")
            if ta_min is not None and ta_max is not None:
                avg_temp = (ta_min + ta_max) / 2
                record = {
                    "날짜": (datetime.now() + timedelta(days=day)).strftime("%Y-%m-%d"),
                    "지역": region,
                    "avg_temp": avg_temp,
                }
                records.append(record)

    return pd.DataFrame(records, columns=["날짜", "지역", "avg_temp"])

# ===== 1) 지역 정의 =====
region_point = {
    '서울특별시': ('61','126'),
    '인천광역시': ('54','125'),
    '경기도': ('60','121'),
    '대전광역시': ('68','100'),
    '충청남도': ('65','110')
}
region_code = {
    '서울특별시': '11B10101',
    '인천광역시': '11B20201',
    '경기도': '11B20601',
    '대전광역시': '11C20401',
    '충청남도': '11C20101'
}

# ===== 2) 단기/중기 데이터 호출 =====
short_df = fetch_short_term_forecast(base_date=datetime.now().strftime("%Y%m%d"), base_time="0200", region_point=region_point)
mid_df = fetch_mid_term_forecast(base_date=datetime.now().strftime("%Y%m%d"), region_code=region_code)

# ===== 3) 단기 + 중기 통합 및 중복 제거 =====
combined_df = pd.concat([short_df, mid_df], ignore_index=True)

# 날짜 형식 완전 통일
combined_df["날짜"] = pd.to_datetime(combined_df["날짜"]).dt.strftime("%Y-%m-%d")

# 날짜 + 지역 기준 중복 제거 (단기 데이터 우선)
final_df = combined_df.drop_duplicates(subset=["날짜", "지역"], keep="first").reset_index(drop=True)

# 지역 + 날짜 정렬
final_df = final_df.sort_values(by=['지역','날짜']).reset_index(drop=True)

# avg_temp 소수점 첫째 자리 유효
final_df['avg_temp'] = np.round(final_df['avg_temp'], 1)

# 컬럼 순서 정렬
final_df = final_df[['날짜','지역','avg_temp']]

# 기존 기온 데이터 업로드
base_temp = pd.read_excel("./data/recent_temp.xlsx")
print("기존 기온 데이터 업로드 완료")

# 기온 데이터 업데이트
temp_data = pd.concat([base_temp, final_df], axis=0).reset_index(drop=True)
temp_data['날짜'] = pd.to_datetime(temp_data['날짜'], format='%Y-%m-%d', errors='coerce')
print("기온 데이터 업데이트 완료")

# 중복 제거
temp_data = temp_data.drop_duplicates(subset=['날짜','지역'], keep="last").reset_index(drop=True)
print("기온 데이터 중복 제거")

# 신규 기온 데이터 저장
temp_data = temp_data.sort_values(by=['지역','날짜'])
temp_data.to_excel("./data/recent_temp.xlsx", index=False)
print("신규 기온 데이터 저장 완료")

display(temp_data.tail(7))

# 날짜 별 기온 데이터 merge
## data의 지역명에서 공백 전까지만 추출
data["지역_key"] = data["지역"].str.split().str[0]
temp_data["지역_key"] = temp_data["지역"].str.split().str[0]

## 병합 (지역_key + 날짜 기준)
data = pd.merge(
    data,
    temp_data[["지역_key", "날짜", "avg_temp"]],
    on=["지역_key", "날짜"],
    how="left"
)

## 컬럼 정리
data = data.drop(columns=["지역_key"])
data = data.sort_values(by=['지역', '날짜']).reset_index(drop=True)
print("기온 데이터 병합 완료!")

display(data)


AutoContainerGeneration version: 1
Environment: prod-daas-dw-cluster.cluster-cgrjlsurd8vz.ap-northeast-2.rds.amazonaws.com
SSH 터널 연결 성공
쿼리 실행 완료
최신 DF 생성 완료
기존 데이터 업로드 완료
데이터 업데이트 완료
데이터 중복 제거
신규 데이터 저장 완료
학습 데이터 구축 완료!
----------------------------------------------------------------------


중기예보 수집: 100%|██████████| 5/5 [00:03<00:00,  1.49it/s]


기존 기온 데이터 업로드 완료
기온 데이터 업데이트 완료
기온 데이터 중복 제거
신규 기온 데이터 저장 완료


,날짜,지역,avg_temp
4373,2026-05-22,충청남도,15.0
4374,2026-05-23,충청남도,20.0
4375,2026-05-24,충청남도,20.5
4376,2026-05-25,충청남도,21.0
4377,2026-05-26,충청남도,21.5
4378,2026-05-27,충청남도,21.0
4379,2026-05-28,충청남도,21.0


기온 데이터 병합 완료!


,지역,날짜,총 물량,지그재그 당일배송,카카오모빌리티,부스터스,other_seller,avg_temp
0,경기도 고양시 덕양구,2024-01-02,48,33,0,0,15,1.9
1,경기도 고양시 덕양구,2024-01-03,48,42,0,0,6,0.5
2,경기도 고양시 덕양구,2024-01-04,61,52,0,0,9,-0.2
3,경기도 고양시 덕양구,2024-01-05,53,47,0,0,6,3.5
4,경기도 고양시 덕양구,2024-01-06,27,27,0,0,0,-0.4
...,...,...,...,...,...,...,...,...
50535,충청북도 청주시 흥덕구,2026-05-14,167,141,0,3,23,NaN
50536,충청북도 청주시 흥덕구,2026-05-15,143,122,0,1,20,NaN
50537,충청북도 청주시 흥덕구,2026-05-16,115,106,0,0,9,NaN
50538,충청북도 청주시 흥덕구,2026-05-17,87,87,0,0,0,NaN


In [18]:
### Feature & Modeling
import numpy as np
import pandas as pd
import holidays
from tqdm import tqdm
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

# =====================================
# 기온 체감도(Temperature Shift) 함수
# =====================================
def add_temp_shift_features(df, date_col="날짜", temp_col="avg_temp"):
    df = df.copy().sort_values(date_col)

    temp_ma10 = df[temp_col].rolling(7, min_periods=1).min()  # 최근 7일 rolling min
    diff = temp_ma10 - df[temp_col]  # 현재 기온과의 차이
    df["temp_shift"] = diff.abs()  # 절댓값
    df["temp_shift_sign"] = np.sign(diff)  # 방향

    df[["temp_shift", "temp_shift_sign"]] = df[["temp_shift", "temp_shift_sign"]].fillna(0.0)
    return df


# =====================================
# (지역별) 타겟별 lag/rolling 피처 생성
# =====================================
def add_target_lag_rolling_features_by_group(
    df,
    target_col,
    group_col="지역",
    lags=(1, 7, 14, 21, 28),
    roll_windows=(7, 14, 21, 28),
):
    df = df.copy()

    if group_col not in df.columns:
        raise KeyError(f"'{group_col}' 컬럼이 없습니다. 현재 컬럼: {df.columns.tolist()}")
    if "날짜" not in df.columns:
        raise KeyError(f"'날짜' 컬럼이 없습니다. 현재 컬럼: {df.columns.tolist()}")

    df = df.sort_values([group_col, "날짜"]).reset_index(drop=True)

    # 미래처럼 target이 없으면 컬럼만 생성
    if target_col not in df.columns:
        for lag in lags:
            df[f"{target_col}_lag_{lag}"] = np.nan
        for w in roll_windows:
            df[f"{target_col}_roll_mean_{w}"] = np.nan
            df[f"{target_col}_roll_std_{w}"] = np.nan
        return df

    # lag
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df.groupby(group_col)[target_col].shift(lag)

    # rolling (shifted)
    shifted = df.groupby(group_col)[target_col].shift(1)

    for w in roll_windows:
        df[f"{target_col}_roll_mean_{w}"] = (
            shifted.groupby(df[group_col])
            .rolling(w, min_periods=max(2, w // 4))
            .mean()
            .reset_index(level=0, drop=True)
        )
        df[f"{target_col}_roll_std_{w}"] = (
            shifted.groupby(df[group_col])
            .rolling(w, min_periods=max(2, w // 4))
            .std()
            .reset_index(level=0, drop=True)
        )

    return df


# =====================================
# 전처리 함수 (지역별)
# =====================================
def preprocess_features_region(
    df,
    group_col="지역",
    manual_total_spike_up_mild_dates=None,
    manual_total_spike_up_strong_dates=None,
    manual_total_spike_down_dates=None,
    manual_zig_spike_up_mild_dates=None,
    manual_zig_spike_up_strong_dates=None,
    manual_zig_spike_down_dates=None,
):
    df = df.copy()
    df["날짜"] = pd.to_datetime(df["날짜"]).dt.normalize()
    df = df.sort_values(["날짜"] if group_col not in df.columns else [group_col, "날짜"]).reset_index(drop=True)

    # ===============================
    # 1) 날짜 기본 피처
    # ===============================
    df["month"] = df["날짜"].dt.month
    df["dow"] = df["날짜"].dt.weekday
    df["weekend"] = (df["dow"].isin([5, 6])).astype(int)
    df["week_of_year"] = df["날짜"].apply(lambda x: x.isocalendar().week)

    df["month"] = pd.Categorical(df["month"], categories=list(range(1, 13)))
    df["dow"] = pd.Categorical(df["dow"], categories=list(range(0, 7)))

    month_dummies = pd.get_dummies(df["month"], prefix="month").astype(int)
    dow_dummies = pd.get_dummies(df["dow"], prefix="dow").astype(int)
    df = pd.concat([df, month_dummies, dow_dummies], axis=1)

    df["month"] = df["month"].astype(int)
    df["dow"] = df["dow"].astype(int)

    # event
    event_periods = [
        ("2025-03-24", "2025-04-08"),
        ("2025-06-01", "2025-07-01"),
        ("2025-09-22", "2025-10-05"),
        ("2025-10-21", "2025-10-21"),
        ("2025-11-17", "2025-12-02"),
        ("2025-12-15", "2025-12-29"),
        ("2026-03-23", "2026-04-06"),
    ]
    df["is_event"] = 0
    for start, end in event_periods:
        mask = (df["날짜"] >= pd.to_datetime(start)) & (df["날짜"] <= pd.to_datetime(end))
        df.loc[mask, "is_event"] = 1

    # ===============================
    # 2) 지역별 auto spike
    # ===============================
    auto_cols = [c for c in ["총 물량", "지그재그 당일배송"] if c in df.columns]

    for c in ["총 물량", "지그재그 당일배송"]:
        df[f"{c}_spike_up_mild"] = 0
        df[f"{c}_spike_up_strong"] = 0
        df[f"{c}_spike_down"] = 0

    if group_col in df.columns:
        group_keys = df[group_col].dropna().unique().tolist()
    else:
        group_keys = [None]

    for col in auto_cols:
        trend_col = f"{col}_trend"
        resid_col = f"{col}_resid_ratio"
        df[trend_col] = np.nan
        df[resid_col] = np.nan

        for g in group_keys:
            mask_g = np.ones(len(df), dtype=bool) if g is None else (df[group_col] == g)

            # 동일 요일 baseline
            for d in range(7):
                mask_d = mask_g & (df["dow"] == d)
                idx_d = df.index[mask_d]
                if len(idx_d) == 0:
                    continue

                s = df.loc[idx_d, col].sort_index()
                baseline = s.shift(1).rolling(window=3, min_periods=2).mean()
                resid = (s - baseline) / baseline

                df.loc[idx_d, trend_col] = baseline
                df.loc[idx_d, resid_col] = resid

            # 평일/주말 threshold
            for wk_flag in [0, 1]:
                mask_wk = mask_g & (df["weekend"] == wk_flag) & df[resid_col].notnull()
                if mask_wk.sum() == 0:
                    continue

                if wk_flag == 0:
                    upper1, upper2, lower1 = 0.10, 0.20, -0.15
                else:
                    upper1, upper2, lower1 = 0.20, 0.25, -0.10

                mask_up_mild = mask_wk & (df[resid_col] >= upper1) & (df[resid_col] < upper2)
                mask_up_strong = mask_wk & (df[resid_col] >= upper2)
                mask_down = mask_wk & (df[resid_col] <= lower1)

                df.loc[mask_up_mild, f"{col}_spike_up_mild"] = 1
                df.loc[mask_up_strong, f"{col}_spike_up_strong"] = 1
                df.loc[mask_down, f"{col}_spike_down"] = 1

    # ===============================
    # 3) manual spike (날짜 기준, 모든 지역 동일 적용)
    # ===============================
    manual_total_spike_up_mild_dates = manual_total_spike_up_mild_dates or []
    manual_total_spike_up_strong_dates = manual_total_spike_up_strong_dates or []
    manual_total_spike_down_dates = manual_total_spike_down_dates or []
    manual_zig_spike_up_mild_dates = manual_zig_spike_up_mild_dates or []
    manual_zig_spike_up_strong_dates = manual_zig_spike_up_strong_dates or []
    manual_zig_spike_down_dates = manual_zig_spike_down_dates or []

    def _apply_manual(spike_info, colname):
        if not spike_info:
            return
            
        # 1. 리스트로 들어오면 전체 지역에 일괄 적용 (기존 방식)
        if isinstance(spike_info, list):
            dts = pd.to_datetime(spike_info).normalize()
            df.loc[df["날짜"].isin(dts), colname] = 1
            
        # 2. 딕셔너리로 들어오면 해당 날짜의 특정 지역만 적용
        elif isinstance(spike_info, dict):
            for date_str, regions in spike_info.items():
                dt = pd.to_datetime(date_str).normalize()
                df.loc[(df["날짜"] == dt) & (df["지역"].isin(regions)), colname] = 1

    _apply_manual(manual_total_spike_up_mild_dates, "총 물량_spike_up_mild")
    _apply_manual(manual_total_spike_up_strong_dates, "총 물량_spike_up_strong")
    _apply_manual(manual_total_spike_down_dates, "총 물량_spike_down")

    _apply_manual(manual_zig_spike_up_mild_dates, "지그재그 당일배송_spike_up_mild")
    _apply_manual(manual_zig_spike_up_strong_dates, "지그재그 당일배송_spike_up_strong")
    _apply_manual(manual_zig_spike_down_dates, "지그재그 당일배송_spike_down")

    # ===============================
    # 4) holiday / pre_holiday_peak
    # ===============================
    years = sorted(df["날짜"].dt.year.unique().tolist())
    kr_holidays = holidays.KR(years=years)

    df["holiday"] = df["날짜"].isin(kr_holidays).astype(int)
    df.loc[df["weekend"] == 1, "holiday"] = 0

    pre_holiday_map = {}
    for hd in kr_holidays:
        found_days = []
        offset = 1
        while len(found_days) < 2 and offset <= 7:
            prev_day = hd - pd.Timedelta(days=offset)
            if prev_day.weekday() not in [5, 6]:
                found_days.append(prev_day)
            offset += 1

        if len(found_days) >= 1:
            pre_holiday_map[found_days[0]] = 1
        if len(found_days) >= 2:
            if (found_days[0] - found_days[1]).days == 1:
                if pre_holiday_map.get(found_days[1], 0) != 1:
                    pre_holiday_map[found_days[1]] = 2

    df["pre_holiday_peak"] = df["날짜"].map(pre_holiday_map).fillna(0).astype(int)
    df.loc[df["holiday"] == 1, "pre_holiday_peak"] = 0

    post_holiday_map = {}
    for hd in kr_holidays:
        offset = 1
        while offset <= 7:
            next_day = hd + pd.Timedelta(days=offset)

            if next_day.weekday() not in [5, 6]:
                post_holiday_map[next_day] = 1
                break

            offset += 1

    df["post_holiday_peak"] = df["날짜"].map(post_holiday_map).fillna(0).astype(int)
    df.loc[df["holiday"] == 1, "post_holiday_peak"] = 0

    # =========================================
    # 5) 미운영 일자 holiday 제거
    # =========================================
    exclude_dates = pd.to_datetime(
        ["2024-02-09", "2024-02-10", "2024-02-11",
         "2024-08-17",
         "2024-09-16", "2024-09-17", "2024-09-18",
         "2025-01-28", "2025-01-29", "2025-01-30",
         "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
         "2026-02-16", "2026-02-17", "2026-02-18"]
    ).normalize()
    df.loc[df["날짜"].isin(exclude_dates), "holiday"] = 0

    # ===============================
    # 6) temp shift
    # ===============================
    if "avg_temp" in df.columns:
        df = add_temp_shift_features(df, temp_col="avg_temp")
        df = df.drop(columns=["avg_temp"], errors="ignore")

    # ===============================
    # 7) lag/rolling (지역별)
    # ===============================
    df = add_target_lag_rolling_features_by_group(
        df,
        "총 물량",
        group_col=group_col,
        lags=(1, 7, 14, 21, 28),
        roll_windows=(7, 14, 21, 28),
    )
    df = add_target_lag_rolling_features_by_group(
        df,
        "지그재그 당일배송",
        group_col=group_col,
        lags=(1, 7, 14, 21, 28),
        roll_windows=(7, 14, 21, 28),
    )

    # ===============================
    # 8) intermediate drop
    # ===============================
    drop_cols = []
    for col in auto_cols:
        drop_cols += [f"{col}_trend", f"{col}_resid_ratio"]

    drop_targets = ["카카오모빌리티", "부스터스", "other_seller"]
    drop_cols += drop_targets

    df = df.drop(columns=drop_cols, errors="ignore")
    return df


# =====================================
# early stopping best_iteration 예측
# =====================================
def _xgb_predict_with_best_iteration(model: XGBRegressor, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "best_iteration") and model.best_iteration is not None:
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            return model.predict(X)
    return model.predict(X)


def _safe_best_iteration(xgb_model):
    if xgb_model is None:
        return None
    bi = getattr(xgb_model, "best_iteration", None)
    if bi is not None:
        return int(bi)
    try:
        booster = xgb_model.get_booster()
        bi2 = getattr(booster, "best_iteration", None)
        return int(bi2) if bi2 is not None else None
    except Exception:
        return None


def _ensure_columns(df: pd.DataFrame, cols: list, fill_value=0.0) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = fill_value
    return df


# =====================================
# Custom Loss Function
# =====================================
def weighted_mse(alpha=1):
    def weighted_mse_fixed(label, pred):
        residual = (label - pred).astype("float")
        grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
        hess = np.where(residual > 0, 2 * alpha, 2.0)
        return grad, hess
    return weighted_mse_fixed


# =====================================
# 지역별 학습/검증/예측
# =====================================
def train_validate_forecast_region_xgb(
    train_proc_region: pd.DataFrame,
    forecast_total_df: pd.DataFrame,
    forecast_days=7,
    valid_days=7,
    raw_data: pd.DataFrame = None,
    temp_future_df: pd.DataFrame = None,
    group_col="지역",
    future_manual_total_spike_up_mild_dates=None,
    future_manual_total_spike_up_strong_dates=None,
    future_manual_total_spike_down_dates=None,
    future_manual_zig_spike_up_mild_dates=None,
    future_manual_zig_spike_up_strong_dates=None,
    future_manual_zig_spike_down_dates=None,
    full_extra_trees=10,
):
    TOTAL = "총 물량"
    ZIG = "지그재그 당일배송"
    RANDOM_STATE = 42
    MIN_TRAIN_SAMPLES = 50

    if raw_data is None:
        raise ValueError("⚠️ raw_data(날짜+avg_temp 포함)를 넘겨줘야 future temp 생성이 가능합니다.")

    # -------------------------
    # 전체물동량(날짜별) 피처 구성
    # -------------------------
    total_item = train_proc_region.groupby("날짜")[TOTAL].sum().reset_index()
    total_item.rename(columns={TOTAL: "전체물동량"}, inplace=True)

    ft = forecast_total_df.copy()
    ft["날짜"] = pd.to_datetime(ft["날짜"]).dt.normalize()
    ft = ft.rename(columns={"pred_총물량": "전체물동량"})

    total_item["날짜"] = pd.to_datetime(total_item["날짜"]).dt.normalize()
    total_item = pd.concat([total_item, ft[["날짜", "전체물동량"]]], ignore_index=True)
    total_item = total_item.drop_duplicates(subset=["날짜"], keep="last").sort_values("날짜").reset_index(drop=True)

    # train merge
    train_proc_region = train_proc_region.copy()
    train_proc_region["날짜"] = pd.to_datetime(train_proc_region["날짜"]).dt.normalize()
    train_proc_region = pd.merge(train_proc_region, total_item, how="left", on="날짜")

    # temp source (global)
    temp_source = raw_data[["날짜", "avg_temp"]].copy()
    temp_source["날짜"] = pd.to_datetime(temp_source["날짜"]).dt.normalize()

    if temp_future_df is not None:
        tfd = temp_future_df.copy()
        tfd["날짜"] = pd.to_datetime(tfd["날짜"]).dt.normalize()
        tfd = tfd[["날짜", "avg_temp"]].copy()
    else:
        tfd = None

    # -------------------------
    # fallback helper
    # -------------------------
    def _make_short_history_fallback_forecast(df_region, forecast_total_df, group_col="지역"):
        """
        신규/짧은 이력 지역 fallback:
        마지막 실제값 기준 전체 대비 점유율로 미래 7일을 배분
        """
        region_name = df_region[group_col].iloc[0]
        hist = df_region.sort_values("날짜").copy()
        last_row = hist.iloc[-1]
        last_date = pd.to_datetime(last_row["날짜"]).normalize()

        total_on_last_date = train_proc_region.loc[
            train_proc_region["날짜"] == last_date, TOTAL
        ].sum()

        zig_on_last_date = train_proc_region.loc[
            train_proc_region["날짜"] == last_date, ZIG
        ].sum()

        total_share = float(last_row[TOTAL]) / float(total_on_last_date) if total_on_last_date > 0 else 0.0
        zig_share = float(last_row[ZIG]) / float(zig_on_last_date) if zig_on_last_date > 0 else 0.0

        future_fallback = forecast_total_df.copy()
        future_fallback["날짜"] = pd.to_datetime(future_fallback["날짜"]).dt.normalize()
        future_fallback[group_col] = region_name

        future_fallback["pred_총물량"] = np.round(
            future_fallback["pred_총물량"] * total_share
        ).clip(lower=0)

        if "pred_지그재그당일배송" in future_fallback.columns:
            future_fallback["pred_지그재그당일배송"] = np.round(
                future_fallback["pred_지그재그당일배송"] * zig_share
            ).clip(lower=0)
        else:
            future_fallback["pred_지그재그당일배송"] = 0.0

        return future_fallback[["날짜", group_col, "pred_총물량", "pred_지그재그당일배송"]]

    # -------------------------
    # 지역 loop
    # -------------------------
    results_valid = []
    results_forecast = []

    regions = train_proc_region[group_col].dropna().unique().tolist()

    for region in tqdm(regions, desc="지역 처리", unit="지역"):
        df_tr = (
            train_proc_region[train_proc_region[group_col] == region]
            .sort_values("날짜")
            .reset_index(drop=True)
            .copy()
        )

        # 신규/짧은 이력 지역 fallback 처리
        if len(df_tr) <= valid_days + 28:
            print(f"⚠️ {region}: 학습 이력이 부족하여 fallback 예측 사용")

            fallback_forecast = _make_short_history_fallback_forecast(
                df_region=df_tr,
                forecast_total_df=forecast_total_df,
                group_col=group_col,
            )
            results_forecast.append(fallback_forecast)
            continue

        split_idx = len(df_tr) - valid_days
        train_df = df_tr.iloc[:split_idx].copy()
        valid_df = df_tr.iloc[split_idx:].copy()

        last_date = df_tr["날짜"].max()
        future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days)
        future_df = pd.DataFrame({"날짜": future_dates})
        future_df["날짜"] = pd.to_datetime(future_df["날짜"]).dt.normalize()
        future_df[group_col] = region

        # --- future avg_temp 구성 ---
        tail_temp = temp_source[temp_source["날짜"] <= last_date].tail(14)

        if tfd is not None:
            future_temp_part = pd.merge(future_df, tfd, on="날짜", how="left")
            last_temp = float(tail_temp["avg_temp"].iloc[-1]) if len(tail_temp) > 0 else 0.0
            future_temp_part["avg_temp"] = future_temp_part["avg_temp"].fillna(last_temp)
        else:
            last_temp = float(tail_temp["avg_temp"].iloc[-1]) if len(tail_temp) > 0 else 0.0
            future_temp_part = future_df.copy()
            future_temp_part["avg_temp"] = last_temp

        future_temp_input = pd.concat(
            [tail_temp.assign(**{group_col: region}), future_temp_part],
            ignore_index=True,
        )

        # --- future 전처리 ---
        future_feat_input = preprocess_features_region(
            future_temp_input,
            group_col=group_col,
            manual_total_spike_up_mild_dates=future_manual_total_spike_up_mild_dates,
            manual_total_spike_up_strong_dates=future_manual_total_spike_up_strong_dates,
            manual_total_spike_down_dates=future_manual_total_spike_down_dates,
            manual_zig_spike_up_mild_dates=future_manual_zig_spike_up_mild_dates,
            manual_zig_spike_up_strong_dates=future_manual_zig_spike_up_strong_dates,
            manual_zig_spike_down_dates=future_manual_zig_spike_down_dates,
        )
        future_feat_df = future_feat_input.iloc[-forecast_days:].reset_index(drop=True)

        # 미래에는 ZIG 컬럼 자체가 없을 수 있음 → chain용으로 미리 생성
        if ZIG not in future_feat_df.columns:
            future_feat_df[ZIG] = np.nan

        # 전체물동량 merge
        future_feat_df = pd.merge(future_feat_df, total_item, how="left", on="날짜")

        # -------------------------
        # 피처 풀 구성
        # -------------------------
        all_cols = df_tr.columns.tolist()

        spike_total_cols = [c for c in all_cols if c.startswith(f"{TOTAL}_spike_")]
        spike_zig_cols = [c for c in all_cols if c.startswith(f"{ZIG}_spike_")]

        total_ts_cols = [c for c in all_cols if c.startswith(f"{TOTAL}_lag_") or c.startswith(f"{TOTAL}_roll_")]
        zig_ts_cols = [c for c in all_cols if c.startswith(f"{ZIG}_lag_") or c.startswith(f"{ZIG}_roll_")]

        # (A) zig 모델: TOTAL 관련 spike/ts 제외, 날짜/지역/타겟 drop
        zig_base_drop = ["날짜", group_col, TOTAL, ZIG]
        zig_exclude = spike_total_cols + total_ts_cols
        zig_features = [c for c in all_cols if c not in zig_base_drop + zig_exclude]

        # (B) total 모델: ZIG는 feature로 사용(총 drop에서 제외), zig spike/ts 제외
        total_base_drop = ["날짜", group_col, TOTAL]
        total_exclude = spike_zig_cols + zig_ts_cols
        total_features = [c for c in all_cols if c not in total_base_drop + total_exclude]

        # -------------------------
        # (A) 지그재그 학습/검증 + FULL
        # -------------------------
        X_train_z = train_df[zig_features].fillna(0.0)
        y_train_z = train_df[ZIG].astype(float).values
        X_valid_z = valid_df[zig_features].fillna(0.0)
        y_valid_z = valid_df[ZIG].astype(float).values

        nz_mask_train_z = (y_train_z > 10)
        X_tr_nz_z = X_train_z.loc[nz_mask_train_z]
        y_tr_nz_z = np.log1p(y_train_z[nz_mask_train_z])

        if len(X_tr_nz_z) >= MIN_TRAIN_SAMPLES:
            xgb_val_z = XGBRegressor(
                n_estimators=5000,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
                early_stopping_rounds=100,
            )
            xgb_val_z.fit(
                X_tr_nz_z,
                y_tr_nz_z,
                eval_set=[(X_valid_z, np.log1p(np.clip(y_valid_z, 0, None)))],
                verbose=False,
            )
            pred_valid_z = np.expm1(_xgb_predict_with_best_iteration(xgb_val_z, X_valid_z))
            pred_valid_z = np.clip(pred_valid_z, 0, None)
        else:
            xgb_val_z = None
            mean_log = float(np.mean(y_tr_nz_z)) if len(y_tr_nz_z) > 0 else 0.0
            pred_valid_z = np.full(len(X_valid_z), np.expm1(mean_log))

        # MAPE zig
        maskz = (y_valid_z > 10)
        mape_zig = mean_absolute_percentage_error(y_valid_z[maskz], pred_valid_z[maskz]) if maskz.sum() > 0 \
            else mean_absolute_percentage_error(y_valid_z, pred_valid_z)

        # FULL zig
        best_it_z = _safe_best_iteration(xgb_val_z)
        full_n_estimators_z = 1500 if best_it_z is None else max(50, best_it_z + int(full_extra_trees))

        X_full_z = df_tr[zig_features].fillna(0.0)
        y_full_z = df_tr[ZIG].astype(float).values
        nz_full_z = (y_full_z > 10)
        X_full_nz_z = X_full_z.loc[nz_full_z]
        y_full_nz_z = np.log1p(y_full_z[nz_full_z])

        if len(X_full_nz_z) >= MIN_TRAIN_SAMPLES:
            xgb_full_z = XGBRegressor(
                n_estimators=full_n_estimators_z,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            xgb_full_z.fit(X_full_nz_z, y_full_nz_z, verbose=False)
        else:
            xgb_full_z = None

        # -------------------------
        # (B) 총물량 학습/검증 (검증은 chain: ZIG를 pred_valid_z로 주입)
        # -------------------------
        X_train_t = train_df[total_features].fillna(0.0)
        y_train_t = train_df[TOTAL].astype(float).values

        nz_mask_train_t = (y_train_t > 10)
        X_tr_nz_t = X_train_t.loc[nz_mask_train_t]
        y_tr_nz_t = np.log1p(y_train_t[nz_mask_train_t])

        X_valid_t_chain = valid_df[total_features].copy()
        if ZIG not in X_valid_t_chain.columns:
            X_valid_t_chain[ZIG] = 0.0
        X_valid_t_chain[ZIG] = pred_valid_z
        X_valid_t_chain = X_valid_t_chain.fillna(0.0)

        y_valid_t = valid_df[TOTAL].astype(float).values

        if len(X_tr_nz_t) >= MIN_TRAIN_SAMPLES:
            xgb_val_t = XGBRegressor(
                n_estimators=5000,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
                early_stopping_rounds=100,
            )
            xgb_val_t.fit(
                X_tr_nz_t,
                y_tr_nz_t,
                eval_set=[(X_valid_t_chain, np.log1p(np.clip(y_valid_t, 0, None)))],
                verbose=False,
            )
            pred_valid_t = np.expm1(_xgb_predict_with_best_iteration(xgb_val_t, X_valid_t_chain))
            pred_valid_t = np.clip(pred_valid_t, 0, None)
        else:
            xgb_val_t = None
            mean_log = float(np.mean(y_tr_nz_t)) if len(y_tr_nz_t) > 0 else 0.0
            pred_valid_t = np.full(len(X_valid_t_chain), np.expm1(mean_log))

        # MAPE total
        maskt = (y_valid_t > 10)
        mape_total = mean_absolute_percentage_error(y_valid_t[maskt], pred_valid_t[maskt]) if maskt.sum() > 0 \
            else mean_absolute_percentage_error(y_valid_t, pred_valid_t)

        # FULL total
        best_it_t = _safe_best_iteration(xgb_val_t)
        full_n_estimators_t = 1500 if best_it_t is None else max(50, best_it_t + int(full_extra_trees))

        X_full_t = df_tr[total_features].fillna(0.0)
        y_full_t = df_tr[TOTAL].astype(float).values
        nz_full_t = (y_full_t > 10)
        X_full_nz_t = X_full_t.loc[nz_full_t]
        y_full_nz_t = np.log1p(y_full_t[nz_full_t])

        if len(X_full_nz_t) >= MIN_TRAIN_SAMPLES:
            xgb_full_t = XGBRegressor(
                n_estimators=full_n_estimators_t,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            xgb_full_t.fit(X_full_nz_t, y_full_nz_t, verbose=False)
        else:
            xgb_full_t = None

        # -------------------------
        # 미래 예측: 재귀(ts overwrite) + chain (ZIG -> TOTAL)
        # -------------------------
        def _overwrite_ts_features(row_df: pd.DataFrame, tgt: str, buf: list):
            row = row_df.copy()
            for lag in (1, 7, 14, 21, 28):
                row[f"{tgt}_lag_{lag}"] = buf[-lag] if len(buf) >= lag else 0.0
            for w in (7, 14, 21, 28):
                tail = buf[-w:] if len(buf) >= w else buf
                if len(tail) >= 2:
                    row[f"{tgt}_roll_mean_{w}"] = float(np.mean(tail))
                    row[f"{tgt}_roll_std_{w}"] = float(np.std(tail, ddof=1))
                else:
                    row[f"{tgt}_roll_mean_{w}"] = 0.0
                    row[f"{tgt}_roll_std_{w}"] = 0.0
            return row

        zig_buf = df_tr[ZIG].astype(float).tolist()
        total_buf = df_tr[TOTAL].astype(float).tolist()

        # (1) 미래 ZIG 예측
        if xgb_full_z is None or len(zig_buf) == 0:
            fallback = float(np.mean(zig_buf[-7:])) if len(zig_buf) >= 7 else (float(np.mean(zig_buf)) if len(zig_buf) else 0.0)
            future_pred_zig = [fallback] * forecast_days
        else:
            preds = []
            for i in range(forecast_days):
                row = future_feat_df.iloc[[i]].copy()
                row = _overwrite_ts_features(row, ZIG, zig_buf)

                row = _ensure_columns(row, zig_features, fill_value=0.0)
                X_row = row[zig_features].fillna(0.0)

                pred_i = float(np.expm1(_xgb_predict_with_best_iteration(xgb_full_z, X_row))[0])
                pred_i = max(0.0, pred_i)

                preds.append(pred_i)
                zig_buf.append(pred_i)
            future_pred_zig = preds

        # (2) 미래 TOTAL 예측: ZIG를 예측치로 주입
        if xgb_full_t is None or len(total_buf) == 0:
            fallback = float(np.mean(total_buf[-7:])) if len(total_buf) >= 7 else (float(np.mean(total_buf)) if len(total_buf) else 0.0)
            future_pred_total = [fallback] * forecast_days
        else:
            preds = []
            for i in range(forecast_days):
                row = future_feat_df.iloc[[i]].copy()
                row[ZIG] = float(future_pred_zig[i])
                row = _overwrite_ts_features(row, TOTAL, total_buf)

                row = _ensure_columns(row, total_features, fill_value=0.0)
                X_row = row[total_features].fillna(0.0)

                pred_i = float(np.expm1(_xgb_predict_with_best_iteration(xgb_full_t, X_row))[0])
                pred_i = max(0.0, pred_i)

                preds.append(pred_i)
                total_buf.append(pred_i)
            future_pred_total = preds

        # -------------------------
        # 결과 저장
        # -------------------------
        valid_result = valid_df[["날짜"]].copy()
        valid_result[group_col] = region

        valid_result["true_총물량"] = y_valid_t
        valid_result["pred_총물량"] = pred_valid_t

        valid_result["true_지그재그당일배송"] = y_valid_z
        valid_result["pred_지그재그당일배송"] = pred_valid_z

        valid_result["APE_total"] = np.abs(valid_result["true_총물량"] - valid_result["pred_총물량"]) / np.maximum(
            1e-9, np.abs(valid_result["true_총물량"])
        )
        valid_result["APE_zigzag"] = np.abs(valid_result["true_지그재그당일배송"] - valid_result["pred_지그재그당일배송"]) / np.maximum(
            1e-9, np.abs(valid_result["true_지그재그당일배송"])
        )

        valid_result["MAPE_total_region"] = mape_total
        valid_result["MAPE_zigzag_region"] = mape_zig

        results_valid.append(valid_result)

        forecast_result = future_df[["날짜"]].copy()
        forecast_result[group_col] = region
        forecast_result["pred_총물량"] = future_pred_total
        forecast_result["pred_지그재그당일배송"] = future_pred_zig

        results_forecast.append(forecast_result)

    # -------------------------
    # Summary
    # -------------------------
    valid_all = pd.concat(results_valid, ignore_index=True) if len(results_valid) else pd.DataFrame()
    forecast_all = pd.concat(results_forecast, ignore_index=True) if len(results_forecast) else pd.DataFrame()

    if len(valid_all) > 0:
        mape_summary = (
            valid_all.groupby(group_col)[["APE_total", "APE_zigzag"]]
            .mean()
            .reset_index()
            .rename(columns={"APE_total": "MAPE_total", "APE_zigzag": "MAPE_zigzag"})
        )
    else:
        mape_summary = pd.DataFrame(columns=[group_col, "MAPE_total", "MAPE_zigzag"])

    return valid_all, forecast_all, mape_summary


In [19]:
### Prediction
print(data.columns.tolist())
print(data.head())

['지역', '날짜', '총 물량', '지그재그 당일배송', '카카오모빌리티', '부스터스', 'other_seller', 'avg_temp']
            지역         날짜  총 물량  지그재그 당일배송  카카오모빌리티  부스터스  other_seller  \
0  경기도 고양시 덕양구 2024-01-02    48         33        0     0            15   
1  경기도 고양시 덕양구 2024-01-03    48         42        0     0             6   
2  경기도 고양시 덕양구 2024-01-04    61         52        0     0             9   
3  경기도 고양시 덕양구 2024-01-05    53         47        0     0             6   
4  경기도 고양시 덕양구 2024-01-06    27         27        0     0             0   

   avg_temp  
0       1.9  
1       0.5  
2      -0.2  
3       3.5  
4      -0.4  


In [32]:
# =========================
# 실행
# =========================
# 1) 지역 데이터 전처리
train_proc_region = preprocess_features_region(
    data,
    group_col="지역",
)
# 2) 지역별 학습/예측
valid_all, forecast_all, mape_summary = train_validate_forecast_region_xgb(
    train_proc_region=train_proc_region,
    
    # 수정할 부분: '총 물량' 컬럼 이름을 함수가 기대하는 'pred_총물량'으로 변경해서 전달
    forecast_total_df=forecast_df.rename(columns={'총 물량': 'pred_총물량'}), 
    
    forecast_days=1,
    valid_days=7,
    raw_data=data,        
    temp_future_df=temp_merge_data, 
    group_col="지역",
    
    future_manual_total_spike_down_dates={
        "2026-05-19": ["서울특별시 강남구"],     
    },
    
    # 전체 지역에 적용하고 싶다면 기존처럼 리스트 사용
    future_manual_total_spike_up_mild_dates=[],
    future_manual_total_spike_up_strong_dates=[],
    future_manual_zig_spike_up_mild_dates=[],
    future_manual_zig_spike_up_strong_dates=[],
    future_manual_zig_spike_down_dates=[],
    full_extra_trees=10,
)

print("\n=== 지역별 Total MAPE 요약 ===")
display(mape_summary.sort_values("MAPE_total")[["지역", "MAPE_total"]].head())
display(mape_summary.sort_values("MAPE_total")[["지역", "MAPE_total"]].tail())

print("\n=== 지역별 zigzag MAPE 요약 ===")
display(mape_summary.sort_values("MAPE_zigzag")[["지역", "MAPE_zigzag"]].head())
display(mape_summary.sort_values("MAPE_zigzag")[["지역", "MAPE_zigzag"]].tail())

print("\n=== valid_all 샘플 ===")
display(valid_all.head())

print("\n=== forecast_all 샘플 ===")
display(forecast_all.head())


지역 처리: 100%|██████████| 72/72 [01:52<00:00,  1.56s/지역]


=== 지역별 Total MAPE 요약 ===


,지역,MAPE_total
41,서울특별시 노원구,0.013956
53,서울특별시 용산구,0.017895
22,경기도 용인시 기흥구,0.020064
4,경기도 구리시,0.020068
13,경기도 수원시 영통구,0.020173


,지역,MAPE_total
60,인천광역시 동구,0.124572
70,충청북도 청주시 청원구,0.155844
71,충청북도 청주시 흥덕구,0.208876
69,충청북도 청주시 서원구,0.215310
68,충청북도 청주시 상당구,0.294965



=== 지역별 zigzag MAPE 요약 ===


...

,지역,MAPE_zigzag
60,인천광역시 동구,0.129522
70,충청북도 청주시 청원구,0.142847
71,충청북도 청주시 흥덕구,0.155710
69,충청북도 청주시 서원구,0.190480
68,충청북도 청주시 상당구,0.202358



=== valid_all 샘플 ===


,날짜,지역,true_총물량,pred_총물량,true_지그재그당일배송,pred_지그재그당일배송,APE_total,APE_zigzag,MAPE_total_region,MAPE_zigzag_region
0,2026-05-12,경기도 고양시 덕양구,232.0,240.944412,196.0,185.430283,0.038554,0.053927,0.038855,0.033953
1,2026-05-13,경기도 고양시 덕양구,282.0,276.102844,226.0,224.718491,0.020912,0.005670,0.038855,0.033953
2,2026-05-14,경기도 고양시 덕양구,301.0,298.620880,250.0,259.124786,0.007904,0.036499,0.038855,0.033953
3,2026-05-15,경기도 고양시 덕양구,274.0,301.169617,246.0,262.038879,0.099159,0.065199,0.038855,0.033953
4,2026-05-16,경기도 고양시 덕양구,212.0,204.751617,199.0,188.486725,0.034190,0.052831,0.038855,0.033953



=== forecast_all 샘플 ===


,날짜,지역,pred_총물량,pred_지그재그당일배송
0,2026-05-19,경기도 고양시 덕양구,256.135071,206.533813
1,2026-05-19,경기도 고양시 일산동구,238.333527,178.723251
2,2026-05-19,경기도 고양시 일산서구,179.836487,138.511505
3,2026-05-19,경기도 광명시,251.383575,197.558319
4,2026-05-19,경기도 구리시,150.080566,110.116096


In [34]:
## Result Save
# 예측 데이터프레임 저장

# 지역 정렬
forecast_all = forecast_all.sort_values(by=['지역','날짜'])

# 기간에 맞는 파일명으로 저장
forecast_df.to_excel("./Total_results/Total_Forecasting_0518_0519_TEST.xlsx", index=False)
forecast_all.to_excel("./Region_results/Region_Forecasting_0518_0519_TEST.xlsx", index=False)

In [ ]:

#휴무일(기간)인 경우, 아래 코드 이용해서 0 값 넣기
data.loc[(data['날짜']>="2026-02-16") & (data['날짜']<="2026-02-18"), "총 물량"] = 0
data.loc[(data['날짜']>="2026-02-16") & (data['날짜']<="2026-02-18"), "지그재그 당일배송"] = 0
data.loc[(data['날짜']>="2026-02-16") & (data['날짜']<="2026-02-18"), "카카오모빌리티"] = 0
data.loc[(data['날짜']>="2026-02-16") & (data['날짜']<="2026-02-18"), "부스터스"] = 0
data.loc[(data['날짜']>="2026-02-16") & (data['날짜']<="2026-02-18"), "other_seller"] = 0

# 누적 실제 물량 엑셀 파일 저장
data['날짜'] = data['날짜'].astype(str)
data[(data['날짜']>="2025-10-11") & (data['날짜']<="2026-05-04")].sort_values(by=['지역','날짜']).reset_index(drop=True).to_excel("./valid.xlsx",index=False)
# 누적 예측 물량 엑셀 파일 불러오기
data1 = pd.read_excel("./Region_results/Region_Forecasting_1010_1016.xlsx")
data2 = pd.read_excel("./Region_results/Region_Forecasting_1015_1021.xlsx")
data3 = pd.read_excel("./Region_results/Region_Forecasting_1022_1028.xlsx")
data4 = pd.read_excel("./Region_results/Region_Forecasting_1029_1104.xlsx")
data5 = pd.read_excel("./Region_results/Region_Forecasting_1105_1111.xlsx")
data6 = pd.read_excel("./Region_results/Region_Forecasting_1112_1118.xlsx")
data7 = pd.read_excel("./Region_results/Region_Forecasting_1119_1125.xlsx")
data8 = pd.read_excel("./Region_results/Region_Forecasting_1126_1202.xlsx")
data9 = pd.read_excel("./Region_results/Region_Forecasting_1203_1209.xlsx")
data10 = pd.read_excel("./Region_results/Region_Forecasting_1210_1216.xlsx")
data11 = pd.read_excel("./Region_results/Region_Forecasting_1217_1225.xlsx")
data12 = pd.read_excel("./Region_results/Region_Forecasting_1226_1230.xlsx")
data13 = pd.read_excel("./Region_results/Region_Forecasting_1231_0106.xlsx")
data14 = pd.read_excel("./Region_results/Region_Forecasting_0107_0113.xlsx")
data15 = pd.read_excel("./Region_results/Region_Forecasting_0114_0120.xlsx")
data16 = pd.read_excel("./Region_results/Region_Forecasting_0121_0127.xlsx")
data17 = pd.read_excel("./Region_results/Region_Forecasting_0128_0203.xlsx")
data18 = pd.read_excel("./Region_results/Region_Forecasting_0204_0210.xlsx")
data19 = pd.read_excel("./Region_results/Region_Forecasting_0211_0220.xlsx")
data20 = pd.read_excel("./Region_results/Region_Forecasting_0221_0224.xlsx")
data21 = pd.read_excel("./Region_results/Region_Forecasting_0225_0303.xlsx")


#0306 인수인계 받은 후
data22 = pd.read_excel("./Region_results/Region_Forecasting_0304_0310.xlsx")
data23 = pd.read_excel("./Region_results/Region_Forecasting_0311_0317.xlsx")
data24 = pd.read_excel("./Region_results/Region_Forecasting_0318_0324.xlsx")
data25 = pd.read_excel("./Region_results/Region_Forecasting_0325_0331.xlsx") 
data26 = pd.read_excel("./Region_results/Region_Forecasting_0401_0407.xlsx") 
data27 = pd.read_excel("./Region_results/Region_Forecasting_0408_0414.xlsx")
data28 = pd.read_excel("./Region_results/Region_Forecasting_0415_0421.xlsx")
data29 = pd.read_excel("./Region_results/Region_Forecasting_0422_0428.xlsx")
data30 = pd.read_excel("./Region_results/Region_Forecasting_0429_0505.xlsx")


data = pd.concat(
    [data1,data2,data3,data4,data5,data6,data7,data8,data9,data10,
     data11,data12,data13,data14,data15,data16,data17,data18,data19,
     data20,data21,data22,data23, data24, data25, data26, data27, data28, data29, data30],
    axis=0
)

data['날짜'] = pd.to_datetime(data['날짜'], errors='coerce')
data = data.drop_duplicates(subset=['날짜','지역'], keep='last')

data = data[
    (data['날짜'] >= pd.Timestamp('2025-10-11')) &
    (data['날짜'] <= pd.Timestamp('2026-05-04'))
]

data = data.sort_values(by=['지역','날짜']).reset_index(drop=True)
data.to_excel("./valid_region.xlsx", index=False)
x = pd.read_excel("./valid.xlsx")
x_ = pd.read_excel("./valid_region.xlsx")
set(x['지역'].unique()) - set(x_['지역'].unique())